# 📊 Análise de Carteira — Ações e FIIs (sinais de entrada/saída)
### Notebook interativo — sem necessidade de programar

Este notebook analisa sua carteira de **ações** e **FIIs** combinando dois tipos de critério:

1. **Fundamentalista** (o ativo está "caro" ou "barato" pelos seus números: P/L, P/VP, ROE, Dividend Yield...)
2. **Técnico** (o preço está numa região historicamente boa de compra/venda: médias móveis, RSI, distância de máximas/mínimas)

No final, cada ativo recebe um **sinal simples**: 🟢 possível entrada, 🟡 neutro/monitorar, 🔴 possível saída/cautela.

> ⚠️ **Aviso importante**: Isso é uma ferramenta educacional de apoio à análise, baseada em regras simples e transparentes.
> **Não é recomendação de investimento.** Os critérios usados são heurísticas (regras de bolso) amplamente conhecidas,
> não uma análise profissional. Sempre complemente com seu próprio julgamento, relatórios de analistas e, se possível,
> um assessor/consultor de investimentos licenciado (CVM) antes de decidir.

**Como usar**: rode as células em ordem (▶ em cada uma, ou "Ambiente de execução → Executar tudo"), preencha os tickers
na Seção 2 e clique nos botões.


In [ ]:
#@title 🔧 1. Preparar o ambiente (clique para rodar)

!uv pip install -q yfinance ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import datetime
import time
from zoneinfo import ZoneInfo
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- Cache local (em memória, válido durante esta sessão do Colab) -----------
# Evita repetir a mesma chamada de rede se você rodar a análise mais de uma vez
# na mesma sessão (ex: Seção 7 e depois Seção 8 pro mesmo ticker, ou clicar em
# "Analisar carteira" de novo sem mudar nada). NÃO persiste entre sessões —
# isso exigiria montar o Google Drive, o que o Colab não faz automaticamente;
# ver Seção 8.1.1 do manual sobre o cache persistente (SQLite) do pacote Python.
_cache_dados = {}
_CACHE_TTL_SEGUNDOS = 30 * 60  # 30 minutos

def _cache_get(chave):
    if chave in _cache_dados:
        valor, salvo_em = _cache_dados[chave]
        if time.time() - salvo_em < _CACHE_TTL_SEGUNDOS:
            return valor
    return None

def _cache_set(chave, valor):
    _cache_dados[chave] = (valor, time.time())

def limpar_cache():
    """Force a busca de dados atualizados, ignorando o que já está em cache
    nesta sessão. Rode esta função manualmente se desconfiar de dado defasado."""
    _cache_dados.clear()
    print(f"Cache limpo às {datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%H:%M')}.")

print("Bibliotecas carregadas.")


## 📝 2. Sua carteira

Preencha os tickers separados por vírgula, em cinco grupos: ações da B3, FIIs da B3, ações americanas,
ETFs da B3 e ETFs americanos. Ações e FIIs/ETFs da B3 usam sufixo `.SA` (ex: `PETR4.SA`, `BOVA11.SA`);
ativos americanos não usam sufixo (ex: `AAPL`, `SPY`).

- **Ações (B3)**: PETR4.SA, ITUB4.SA, WEGE3.SA ...
- **FIIs**: HGLG11.SA, KNRI11.SA, MXRF11.SA ...
- **Ações (EUA)**: AAPL, MSFT, GOOGL ...
- **ETFs (B3)**: BOVA11.SA, IVVB11.SA, SMAL11.SA, ou de renda fixa (ex: IMAB11.SA, B5P211.SA) ...
- **ETFs (EUA)**: SPY, QQQ, VOO ...

As duas caixas de ETF aceitam **qualquer tipo de ETF** (de ações, de renda fixa/juros, multimercado) —
não separamos por classe de ativo dentro de ETF, só por bolsa (B3 x EUA). Isso importa porque a
terminologia do que é distribuído muda: ETFs de ações costumam chamar de "dividendo/rendimento" (quando
distribuem — muitos são "de acumulação" e não distribuem nada, reinvestindo internamente), enquanto ETFs
de renda fixa repassam os juros dos títulos que carregam (às vezes chamado de "cupom"). Para efeito de
cálculo, o notebook trata ambos da mesma forma: soma o que a fonte de dados registrou como provento pago
nos últimos 12 meses, seja qual for o nome técnico.

Os critérios de avaliação são diferentes por grupo. Ações usam P/L, P/VP, ROE e dividend yield, com
limiares próprios para B3 e EUA (o mercado americano historicamente negocia a múltiplos mais altos).
ETFs não têm P/L, P/VP nem ROE — não são empresas, são cestas de ativos — então o critério é só técnico
(tendência, RSI, distância do período) mais yield. Em ambos os casos, ativos sem histórico de distribuição
são classificados como "não aplicável" (ações de crescimento ou ETFs de acumulação), não como sinal
negativo — é comum e esperado nesses perfis.

> ⚠️ **Não suportado: títulos de renda fixa individuais (bonds corporativos, ex: "Citigroup 6,27% 2034")**.
> O Yahoo Finance não tem cotação diária por ticker pra bonds individuais — eles negociam em balcão (OTC),
> identificados por CUSIP, não por um ticker de bolsa. Mesmo que houvesse o dado, os critérios técnicos
> desta ferramenta (RSI, médias móveis) não fazem sentido pra um bond específico, que pode passar dias sem
> negociar — a análise correta seria yield-to-maturity, duration e spread de crédito, algo fora do escopo
> deste protótipo. ETFs de renda fixa (ex: TLT, IEF, AGG, IMAB11.SA) são diferentes e **são suportados**
> normalmente nas caixas de ETF acima, porque negociam como qualquer ETF, com ticker e liquidez diária.


In [ ]:
#@title 📝 2. Digite os tickers da sua carteira

caixa_acoes = widgets.Textarea(
    value='VALE3.SA, ABCB4.SA, BBSE3.SA, BBDC4.SA, LEVE3.SA, POMO4.SA, ITSA4.SA, CMIG4.SA, BBAS3.SA, KLBN11.SA, CXSE3.SA, PSSA3.SA',
    description='Ações (B3):',
    placeholder='Ex: PETR4.SA, ITUB4.SA, WEGE3.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_fiis = widgets.Textarea(
    value='HGCR11.SA, RECR11.SA, KNSC11.SA, CPTS11.SA, VGIP11.SA, KNIP11.SA, BTHF11.SA, VRTA11.SA, BTLG11.SA, HGRE11.SA, HGLG11.SA, XPML11.SA, TRXF11.SA, JURO11.SA, TGAR11.SA',
    description='FIIs:',
    placeholder='Ex: HGLG11.SA, KNRI11.SA, MXRF11.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_acoes_us = widgets.Textarea(
    value='BRK-B',
    description='Ações (EUA):',
    placeholder='Ex: AAPL, MSFT, GOOGL',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_etf_br = widgets.Textarea(
    value='BOVA11.SA, IVVB11.SA',
    description='ETFs (B3):',
    placeholder='Ex: BOVA11.SA, IVVB11.SA, SMAL11.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_etf_us = widgets.Textarea(
    value='QQQ, TLT, IAU, IVV, IEF, AGG, MCHI, EMXC, REMX, XME, IXG, XLE',
    description='ETFs (EUA):',
    placeholder='Ex: SPY, QQQ, VOO',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

periodo_historico = widgets.Dropdown(
    options=['6mo', '1y', '2y', '5y'],
    value='2y',
    description='Histórico p/ análise técnica:',
    style={'description_width': 'initial'}
)

display(widgets.VBox([
    widgets.HTML("<h3>📝 Carteira</h3>"),
    caixa_acoes,
    caixa_fiis,
    caixa_acoes_us,
    caixa_etf_br,
    caixa_etf_us,
    periodo_historico
]))


## ⚙️ 3. Indicadores usados

**Técnicos** (calculados a partir do histórico de preços — Seção 4):
- Preço em relação à média móvel de 200 dias (tendência de longo prazo)
- RSI de 14 dias (sobrecomprado/sobrevendido)
- Distância em relação à faixa alta/baixa do período (percentil 98%/2%, não o máximo/mínimo
  absoluto — isso evita que um único dado ruim do provedor distorça o indicador inteiro)

**Fundamentalistas** (obtidos via Yahoo Finance — Seção 5), critérios diferentes para ações e FIIs:
- **Ações**: P/L, P/VP, ROE, Dividend Yield
- **FIIs**: P/VP, Dividend Yield, liquidez média diária

**Renda passiva** (histórico real de dividendos/rendimentos pagos, Seção 5 e 6):
- Yield efetivo dos últimos 12 meses (soma real paga ÷ preço atual — mais confiável que um yield "de tela")
- Crescimento dos dividendos em relação ao mesmo período do ano anterior
- Regularidade dos pagamentos (FIIs costumam pagar mensalmente, ações 1-2x/ano)

O resultado final traz **três colunas**: um *sinal de timing* (bom momento de comprar/vender), uma
*qualidade da renda* (dividendos crescentes e consistentes ou não) e uma **estratégia sugerida**, que combina
as duas coisas — por exemplo: se o preço subiu muito mas a renda continua saudável, o ativo não precisa ser
mantido a qualquer preço; pode valer a pena vender e recomprar depois, se/quando o preço voltar a um ponto
melhor. Já se a renda estiver piorando, isso pesa mais do que o preço, porque é sinal de que a tese original
pode ter mudado.

Cada indicador soma ou subtrai pontos de uma pontuação simples (Seção 6), que gera os sinais finais.
**Os limites usados são regras de bolso genéricas — ajuste-os na célula de pontuação se quiser refletir sua estratégia.**


In [ ]:
#@title 📈 4. Funções de indicadores técnicos

def calcular_rsi(precos, janela=14):
    delta = precos.diff()
    ganho = delta.clip(lower=0)
    perda = -delta.clip(upper=0)
    media_ganho = ganho.rolling(janela).mean()
    media_perda = perda.rolling(janela).mean()
    rs = media_ganho / media_perda.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def agrupar_datas_em_ciclos(datas, limite_dias=45):
    """Agrupa uma lista de datas em 'ciclos' — datas próximas (menos de `limite_dias`
    de intervalo) ficam no mesmo grupo; um intervalo maior que isso indica que o preço
    se afastou e voltou depois, ou seja, um novo ciclo. Usado para distinguir um ativo
    fazendo mínima/máxima pela primeira vez de um ativo revisitando um patamar que já
    tinha visitado antes (e recuperado no meio)."""
    datas_ordenadas = sorted(datas)
    ciclos = [[datas_ordenadas[0]]]
    for d in datas_ordenadas[1:]:
        if (d - ciclos[-1][-1]).days > limite_dias:
            ciclos.append([d])
        else:
            ciclos[-1].append(d)
    return ciclos

def pregao_provavelmente_aberto(data_ultima_cotacao):
    """Estimativa simples (não oficial) de se o pregão da B3 pode ainda estar em
    andamento no momento em que a análise é rodada. Usada só pra avisar que o preço
    mais recente pode ainda não ser o fechamento definitivo do dia — não substitui o
    calendário oficial da B3 (não considera feriados, por exemplo)."""
    agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo'))
    if data_ultima_cotacao.date() != agora.date():
        return False  # a cotação mais recente já é de um dia anterior (pregão daquele dia encerrado)
    if agora.weekday() >= 5:
        return False  # sábado ou domingo, não tem pregão
    return datetime.time(10, 0) <= agora.time() <= datetime.time(18, 0)

def limpar_outliers_precos(fechamento, janela=21, limite=0.35, iteracoes=3, limite_nivel=0.6):
    """Trata dois tipos de problema de dados, de formas diferentes, NESTA ORDEM:

    1) TICKS ISOLADOS: comparação com a mediana de uma janela local (21 dias), em
       várias passadas. Roda PRIMEIRO de propósito — corrige por interpolação os
       erros pontuais de poucos dias antes de qualquer outra análise. Isso importa
       porque, se um tick isolado no MEIO do histórico for avaliado antes por um
       critério global (etapa 2), ele pode ser confundido com o início de um bloco
       inteiro fora de escala, levando a truncar (descartar) uma quantidade enorme
       de dados bons só por causa de um erro de poucos dias.

    2) BLOCO ANTIGO FORA DE ESCALA (ex: agrupamento/desdobramento de cotas que o
       provedor de dados não ajustou retroativamente): SÓ DEPOIS de (1), comparamos
       cada preço remanescente com a mediana dos ÚLTIMOS 60 DIAS (a cotação atual de
       verdade). Um trecho antigo severamente fora dessa faixa — e que sobreviveu à
       limpeza local porque é consistente com seus próprios vizinhos — é DESCARTADO
       por completo; interpolar seria arriscado quando o bloco é grande. Indicadores
       que precisam de mais dados do que sobrou (ex: MM200) ficam automaticamente
       como 'não disponível' em vez de mostrar um número calculado sobre dado ruim."""
    serie = fechamento.copy()

    total_corrigidos = 0
    for _ in range(iteracoes):
        mediana_movel = serie.rolling(janela, center=True, min_periods=1).median()
        desvio_relativo = (serie - mediana_movel).abs() / mediana_movel
        suspeitos = desvio_relativo > limite
        n_suspeitos = int(suspeitos.sum())
        if n_suspeitos == 0:
            break
        serie[suspeitos] = np.nan
        serie = serie.interpolate().ffill().bfill()
        total_corrigidos += n_suspeitos

    referencia_recente = serie.iloc[-60:].median() if len(serie) >= 60 else serie.median()
    desvio_referencia = (serie - referencia_recente).abs() / referencia_recente
    suspeitos_nivel = desvio_referencia > limite_nivel

    n_truncados = 0
    if suspeitos_nivel.any():
        ultimo_suspeito_pos = int(np.where(suspeitos_nivel.values)[0].max())
        if ultimo_suspeito_pos < len(serie) - 1:
            n_truncados = ultimo_suspeito_pos + 1
            serie = serie.iloc[ultimo_suspeito_pos + 1:]
            total_corrigidos += n_truncados

    return serie, total_corrigidos, n_truncados


def indicadores_tecnicos(ticker, periodo):
    """Baixa histórico e calcula indicadores técnicos de um ativo."""
    chave_cache = f"precos:{ticker}:{periodo}"
    fechamento_bruto_cache = _cache_get(chave_cache)
    if fechamento_bruto_cache is not None:
        fechamento = fechamento_bruto_cache
    else:
        hist = yf.download(ticker, period=periodo, progress=False)
        if hist.empty:
            return None
        fechamento = hist['Close']
        if isinstance(fechamento, pd.DataFrame):
            fechamento = fechamento.squeeze(axis=1)
        fechamento = fechamento.dropna()
        if not fechamento.empty:
            _cache_set(chave_cache, fechamento)

    if len(fechamento) < 30:
        return None

    fechamento, n_outliers, n_truncados = limpar_outliers_precos(fechamento)

    if len(fechamento) < 30:
        return None

    preco_atual = float(fechamento.iloc[-1])
    sma50 = fechamento.rolling(50).mean().iloc[-1] if len(fechamento) >= 50 else np.nan
    sma200 = fechamento.rolling(200).mean().iloc[-1] if len(fechamento) >= 200 else np.nan
    rsi14 = calcular_rsi(fechamento, 14).iloc[-1]
    # Percentil 5%/95%, não mínimo/máximo absolutos, e com folga maior que o normal (2%)
    # de propósito: alguns tickers da B3 têm um BLOCO inteiro de dias com preço numa escala
    # errada no histórico do Yahoo Finance (ex: um agrupamento/desdobramento de cotas que não
    # foi reajustado retroativamente). Como esses dias são consistentes ENTRE SI, o filtro de
    # outliers acima (que compara cada dia com seus vizinhos) não os pega — só um corte por
    # percentil com folga suficiente resolve.
    maxima_periodo = fechamento.quantile(0.95)
    minima_periodo = fechamento.quantile(0.05)
    dist_maxima = (preco_atual - maxima_periodo) / maxima_periodo * 100
    dist_minima = (preco_atual - minima_periodo) / minima_periodo * 100

    return {
        'preco_atual': preco_atual,
        'sma50': float(sma50) if not pd.isna(sma50) else None,
        'sma200': float(sma200) if not pd.isna(sma200) else None,
        'rsi14': float(rsi14) if not pd.isna(rsi14) else None,
        'dist_maxima_pct': float(dist_maxima),
        'dist_minima_pct': float(dist_minima),
        'n_outliers_corrigidos': n_outliers,
        'n_dias_truncados': n_truncados,
        'dias_usados': len(fechamento),
        'data_ultima_cotacao': fechamento.index[-1],
        'historico': fechamento
    }

print("Funções técnicas prontas.")


In [ ]:
#@title 💰 5. Funções de indicadores fundamentalistas

def _info_cache(ticker):
    """Busca yf.Ticker(ticker).info com cache — compartilhado entre ações e
    FIIs, já que ambos leem o mesmo dicionário 'info' por baixo."""
    chave = f"info:{ticker}"
    em_cache = _cache_get(chave)
    if em_cache is not None:
        return em_cache
    try:
        info = yf.Ticker(ticker).info
    except Exception:
        return {}
    if info:
        _cache_set(chave, info)
    return info

def indicadores_fundamentalistas_acao(ticker):
    """Busca múltiplos fundamentalistas de uma ação. Não usamos o campo
    'dividendYield' bruto do Yahoo aqui de propósito: em vários tickers da
    B3 ele vem com escala inconsistente (às vezes já em %, às vezes fração),
    o que gerava valores absurdos como '1224%'. O yield real é calculado à
    parte, em historico_dividendos(), a partir dos dividendos efetivamente
    pagos.

    Para AÇÕES DA B3, P/L, P/VP e ROE vêm preferencialmente do Fundamentus
    (sua base original e mais consolidada — mais confiável que o Yahoo
    Finance para tickers da B3); se indisponível, cai de volta no Yahoo,
    sem quebrar a análise. Dados extras do Fundamentus (ROIC, margem
    líquida, dívida bruta/patrimônio, liquidez corrente) também entram no
    dicionário. Não se aplica a ações dos EUA (Fundamentus cobre só B3)."""
    info = _info_cache(ticker)
    fund = {
        'pl': info.get('trailingPE'),
        'pvp': info.get('priceToBook'),
        'roe': info.get('returnOnEquity'),
        'divida_patrimonio': info.get('debtToEquity'),
    }
    if ticker.upper().endswith('.SA'):  # Fundamentus só cobre B3, não ações EUA
        fundamentus = buscar_fundamentos_acao_fundamentus(ticker)
        if fundamentus is not None:
            if fundamentus.get('pl') is not None:
                fund['pl'] = fundamentus['pl']
            if fundamentus.get('pvp') is not None:
                fund['pvp'] = fundamentus['pvp']
            if fundamentus.get('roe') is not None:
                fund['roe'] = fundamentus['roe']
            fund['roic'] = fundamentus.get('roic')
            fund['margem_liquida'] = fundamentus.get('margem_liquida')
            fund['divida_bruta_patrimonio'] = fundamentus.get('divida_bruta_patrimonio')
            fund['liquidez_corrente'] = fundamentus.get('liquidez_corrente')
    return fund

import io
import re
import urllib.request

_URL_FII_RESULTADO = "https://www.fundamentus.com.br/fii_resultado.php"
_USER_AGENT_FUNDAMENTUS = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
_cache_tabela_fundamentus = None  # cache em memória da tabela inteira (1 chamada por sessão)

def _parse_numero_br_fundamentus(valor):
    """Converte número em formato brasileiro ('1.234,56' ou '10,80%') para
    float. Também aceita valor já convertido para float/int pelo pandas
    (algumas colunas vêm assim, outras não, dependendo de terem '%' junto —
    reprocessar um float como se fosse texto gera erro, ver teste do pacote)."""
    if valor is None:
        return None
    if isinstance(valor, (int, float)):
        return None if (isinstance(valor, float) and pd.isna(valor)) else float(valor)
    t = str(valor).strip()
    if not t or t in ('-', 'N/A', 'nan'):
        return None
    eh_percentual = t.endswith('%')
    t = t.rstrip('%').strip().replace('.', '').replace(',', '.')
    try:
        n = float(t)
    except ValueError:
        return None
    return n / 100 if eh_percentual else n

def _baixar_html_fii_resultado():
    req = urllib.request.Request(_URL_FII_RESULTADO, headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
    with urllib.request.urlopen(req, timeout=20) as resp:
        return resp.read().decode('latin-1')

def buscar_todos_fiis_fundamentus():
    """Busca a tabela completa de FIIs do Fundamentus (site de terceiros —
    fundamentos que o Yahoo Finance não fornece de forma confiável para
    FIIs: P/VP, Dividend Yield, Segmento, FFO Yield, Vacância, Cap Rate).
    Uma chamada de rede para ~500+ fundos, reaproveitada via cache em
    memória para todos os FIIs analisados na mesma sessão."""
    html = _baixar_html_fii_resultado()
    df = pd.read_html(io.StringIO(html), thousands='.', decimal=',')[0]
    colunas = {c: str(c).strip().lower() for c in df.columns}

    def achar(*pistas):
        for col, nome in colunas.items():
            if any(p in nome for p in pistas):
                return col
        return None

    col_papel, col_segmento = achar('papel'), achar('segmento')
    col_pvp, col_dy, col_ffo = achar('p/vp', 'pvp'), achar('dividend'), achar('ffo')
    col_vac, col_cap, col_imoveis = achar('vac'), achar('cap rate'), achar('qtd de im', 'imóveis', 'imoveis')

    resultado = {}
    for _, linha in df.iterrows():
        ticker_bruto = linha.get(col_papel)
        if ticker_bruto is None or (isinstance(ticker_bruto, float) and pd.isna(ticker_bruto)):
            continue
        ticker = re.sub(r'\s+', '', str(ticker_bruto)).upper()
        if not ticker:
            continue
        resultado[ticker] = {
            'segmento': str(linha.get(col_segmento, '')).strip() if col_segmento else '',
            'pvp': _parse_numero_br_fundamentus(linha.get(col_pvp)) if col_pvp else None,
            'dividend_yield_fundamentus': _parse_numero_br_fundamentus(linha.get(col_dy)) if col_dy else None,
            'ffo_yield': _parse_numero_br_fundamentus(linha.get(col_ffo)) if col_ffo else None,
            'vacancia_media': _parse_numero_br_fundamentus(linha.get(col_vac)) if col_vac else None,
            'cap_rate': _parse_numero_br_fundamentus(linha.get(col_cap)) if col_cap else None,
            'qtd_imoveis': (int(_parse_numero_br_fundamentus(linha.get(col_imoveis)))
                             if col_imoveis and _parse_numero_br_fundamentus(linha.get(col_imoveis)) is not None else None),
        }
    return resultado

def buscar_fundamentos_fii_fundamentus(ticker):
    """Busca os fundamentos de um FII específico no Fundamentus, com cache
    em memória da tabela completa. Devolve None em qualquer falha (rede
    indisponível, ticker não encontrado, mudança de layout) — quem chama
    (indicadores_fundamentalistas_fii, abaixo) cai de volta no Yahoo
    Finance nesse caso, sem quebrar a análise."""
    global _cache_tabela_fundamentus
    ticker_normalizado = re.sub(r'\.SA$', '', ticker.strip().upper())
    try:
        if _cache_tabela_fundamentus is None:
            _cache_tabela_fundamentus = buscar_todos_fiis_fundamentus()
        return _cache_tabela_fundamentus.get(ticker_normalizado)
    except Exception:
        return None


def _achar_coluna_acao(colunas, *pistas):
    """Localiza uma coluna por substring, tolerando acento (a tabela de
    ações do Fundamentus tem nomes como 'Mrg. Líq.' e 'Dív.Brut/ Patrim.')."""
    def normalizar(s):
        s = s.lower()
        for de, para in (('í', 'i'), ('î', 'i'), ('ó', 'o'), ('ô', 'o'), ('á', 'a'), ('â', 'a'), ('ã', 'a')):
            s = s.replace(de, para)
        return s
    for col, nome in colunas.items():
        nome_norm = normalizar(nome)
        if any(normalizar(p) in nome_norm for p in pistas):
            return col
    return None

_URL_ACAO_RESULTADO = "https://www.fundamentus.com.br/resultado.php"
_cache_tabela_acoes_fundamentus = None  # cache em memória, independente do cache de FIIs

def _baixar_html_acao_resultado():
    req = urllib.request.Request(_URL_ACAO_RESULTADO, headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
    with urllib.request.urlopen(req, timeout=20) as resp:
        return resp.read().decode('latin-1')

def buscar_todas_acoes_fundamentus():
    """Busca a tabela completa de ações da B3 do Fundamentus — sua base
    original e mais consolidada (mais confiável que o Yahoo Finance para
    tickers da B3). Não cobre ETFs (não são empresas com balanço
    patrimonial próprio) nem ações dos EUA (cobertura restrita à B3)."""
    html = _baixar_html_acao_resultado()
    df = pd.read_html(io.StringIO(html), thousands='.', decimal=',')[0]
    colunas = {c: str(c).strip().lower() for c in df.columns}

    col_papel = _achar_coluna_acao(colunas, 'papel')
    col_pl = _achar_coluna_acao(colunas, 'p/l')
    col_pvp = _achar_coluna_acao(colunas, 'p/vp')
    col_roe = _achar_coluna_acao(colunas, 'roe')
    col_roic = _achar_coluna_acao(colunas, 'roic')
    col_margem = _achar_coluna_acao(colunas, 'mrg.liq', 'mrg. liq', 'margem liq')
    col_divida = _achar_coluna_acao(colunas, 'div.brut', 'dív.brut', 'divida bruta')
    col_liquidez = _achar_coluna_acao(colunas, 'liq.corr', 'liq. corr')

    resultado = {}
    for _, linha in df.iterrows():
        ticker_bruto = linha.get(col_papel)
        if ticker_bruto is None or (isinstance(ticker_bruto, float) and pd.isna(ticker_bruto)):
            continue
        ticker = re.sub(r'\s+', '', str(ticker_bruto)).upper()
        if not ticker:
            continue
        resultado[ticker] = {
            'pl': _parse_numero_br_fundamentus(linha.get(col_pl)) if col_pl else None,
            'pvp': _parse_numero_br_fundamentus(linha.get(col_pvp)) if col_pvp else None,
            'roe': _parse_numero_br_fundamentus(linha.get(col_roe)) if col_roe else None,
            'roic': _parse_numero_br_fundamentus(linha.get(col_roic)) if col_roic else None,
            'margem_liquida': _parse_numero_br_fundamentus(linha.get(col_margem)) if col_margem else None,
            'divida_bruta_patrimonio': _parse_numero_br_fundamentus(linha.get(col_divida)) if col_divida else None,
            'liquidez_corrente': _parse_numero_br_fundamentus(linha.get(col_liquidez)) if col_liquidez else None,
        }
    return resultado

def buscar_fundamentos_acao_fundamentus(ticker):
    """Busca os fundamentos de uma ação B3 específica no Fundamentus, com
    cache em memória. Devolve None em qualquer falha — quem chama
    (indicadores_fundamentalistas_acao) cai de volta no Yahoo Finance."""
    global _cache_tabela_acoes_fundamentus
    ticker_normalizado = re.sub(r'\.SA$', '', ticker.strip().upper())
    try:
        if _cache_tabela_acoes_fundamentus is None:
            _cache_tabela_acoes_fundamentus = buscar_todas_acoes_fundamentus()
        return _cache_tabela_acoes_fundamentus.get(ticker_normalizado)
    except Exception:
        return None


def indicadores_fundamentalistas_fii(ticker):
    """Busca múltiplos relevantes de um FII. P/VP e Dividend Yield vêm
    preferencialmente do Fundamentus (mais confiável para FIIs do que o
    campo 'priceToBook' do Yahoo Finance, frequentemente ausente/impreciso
    para fundos imobiliários); se o Fundamentus estiver indisponível, cai
    de volta no Yahoo Finance, sem quebrar a análise. Dados extras do
    Fundamentus (Segmento, FFO Yield, Vacância, Cap Rate) também entram no
    dicionário, mesmo sem equivalente no Yahoo."""
    info = _info_cache(ticker)
    fund = {
        'pvp': info.get('priceToBook'),
        'volume_medio': info.get('averageVolume'),
    }
    fundamentus = buscar_fundamentos_fii_fundamentus(ticker)
    if fundamentus is not None:
        if fundamentus.get('pvp') is not None:
            fund['pvp'] = fundamentus['pvp']
        fund['dividend_yield_fundamentus'] = fundamentus.get('dividend_yield_fundamentus')
        fund['segmento'] = fundamentus.get('segmento')
        fund['ffo_yield'] = fundamentus.get('ffo_yield')
        fund['vacancia_media'] = fundamentus.get('vacancia_media')
        fund['cap_rate'] = fundamentus.get('cap_rate')
        fund['qtd_imoveis'] = fundamentus.get('qtd_imoveis')
    return fund

def historico_dividendos(ticker, preco_atual):
    """Busca o histórico real de dividendos/rendimentos pagos e calcula
    yield efetivo (12m), crescimento ano contra ano e regularidade dos pagamentos.
    Isso é mais confiável que o campo 'dividendYield' isolado, que só reflete um instante."""
    chave_cache = f"dividendos:{ticker}"
    divs = _cache_get(chave_cache)
    if divs is None:
        try:
            divs = yf.Ticker(ticker).dividends
        except Exception:
            return None
        if divs is None or divs.empty:
            return None
        _cache_set(chave_cache, divs)

    if divs is None or divs.empty:
        return None

    if divs.index.tz is not None:
        divs.index = divs.index.tz_localize(None)

    hoje = divs.index.max()
    ult_12m = divs[divs.index > hoje - pd.Timedelta(days=365)]
    anterior_12m = divs[(divs.index <= hoje - pd.Timedelta(days=365)) &
                         (divs.index > hoje - pd.Timedelta(days=730))]

    soma_12m = float(ult_12m.sum())
    soma_anterior_12m = float(anterior_12m.sum())

    crescimento_yoy = None
    if soma_anterior_12m > 0:
        crescimento_yoy = (soma_12m - soma_anterior_12m) / soma_anterior_12m

    dy_12m_real = (soma_12m / preco_atual) if preco_atual else None

    return {
        'soma_12m': soma_12m,
        'soma_anterior_12m': soma_anterior_12m,
        'crescimento_yoy': crescimento_yoy,
        'dy_12m_real': dy_12m_real,
        'n_pagamentos_12m': int(len(ult_12m)),
    }

print("Funções fundamentalistas prontas.")
print("⚠️ Nem todo FII/ação da B3 tem todos os campos preenchidos no Yahoo Finance —")
print("   quando faltar um dado, o indicador aparece como 'N/D' e simplesmente não pontua.")


In [ ]:
#@title 🧮 6. Motor de pontuação (ajuste os limites aqui se quiser)

def pontuar_acao(tec, fund, div_info):
    pontos = 0
    detalhes = []

    if tec.get('sma200') is not None:
        if tec['preco_atual'] > tec['sma200']:
            pontos += 1; detalhes.append("+1 preço acima da MM200 (tendência de alta)")
        else:
            pontos -= 1; detalhes.append("-1 preço abaixo da MM200 (tendência de baixa)")

    if tec.get('rsi14') is not None:
        if tec['rsi14'] < 40:
            pontos += 1; detalhes.append("+1 RSI < 40 (não sobrecomprado)")
        elif tec['rsi14'] > 70:
            pontos -= 1; detalhes.append("-1 RSI > 70 (sobrecomprado)")

    _minima_periodo = tec['preco_atual'] / (1 + tec['dist_minima_pct'] / 100)
    _maxima_periodo = tec['preco_atual'] / (1 + tec['dist_maxima_pct'] / 100)
    if tec['dist_minima_pct'] < 15:
        pontos += 1
        _verbo_min = 'abaixo da' if tec['preco_atual'] < _minima_periodo else 'próximo da'
        detalhes.append(f"+1 {_verbo_min} mínima do período (atual {tec['preco_atual']:.2f}, mínima {_minima_periodo:.2f})")
    if tec['dist_maxima_pct'] > -5:
        pontos -= 1
        _verbo_max = 'acima da' if tec['preco_atual'] > _maxima_periodo else 'próximo da'
        detalhes.append(f"-1 {_verbo_max} máxima do período (atual {tec['preco_atual']:.2f}, máxima {_maxima_periodo:.2f})")

    pl = fund.get('pl')
    if pl is not None and pl > 0:
        if pl < 15:
            pontos += 1; detalhes.append(f"+1 P/L baixo ({pl:.1f})")
        elif pl > 25:
            pontos -= 1; detalhes.append(f"-1 P/L alto ({pl:.1f})")

    pvp = fund.get('pvp')
    if pvp is not None:
        if pvp < 1.5:
            pontos += 1; detalhes.append(f"+1 P/VP baixo ({pvp:.2f})")
        elif pvp > 4:
            pontos -= 1; detalhes.append(f"-1 P/VP alto ({pvp:.2f})")

    roe = fund.get('roe')
    if roe is not None and roe > 0.12:
        pontos += 1; detalhes.append(f"+1 ROE saudável ({roe*100:.1f}%)")

    # Indicadores exclusivos do Fundamentus (só B3, não disponíveis via Yahoo Finance)
    roic = fund.get('roic')
    if roic is not None and roic > 0.15:
        pontos += 1; detalhes.append(f"+1 ROIC saudável ({roic*100:.1f}%)")

    margem_liquida = fund.get('margem_liquida')
    if margem_liquida is not None and margem_liquida > 0.10:
        pontos += 1; detalhes.append(f"+1 margem líquida saudável ({margem_liquida*100:.1f}%)")

    # Setor financeiro (bancos, seguradoras) tem estrutura de balanço
    # diferente de empresas comuns — o Fundamentus mostra "-" (não aplicável)
    # pra dívida bruta/patrimônio e liquidez corrente nesses casos, o que
    # nossa tabela recebe como 0.00, não como ausente. Trata exatamente 0.0
    # como dado ausente aqui (caso real: BBAS3/Banco do Brasil).
    divida_patrim = fund.get('divida_bruta_patrimonio')
    if divida_patrim is not None and divida_patrim > 0:
        if divida_patrim < 0.5:
            pontos += 1; detalhes.append(f"+1 baixo endividamento (dívida/patrimônio {divida_patrim:.2f})")
        elif divida_patrim > 1.5:
            pontos -= 1; detalhes.append(f"-1 alto endividamento (dívida/patrimônio {divida_patrim:.2f})")

    liquidez_corrente = fund.get('liquidez_corrente')
    if liquidez_corrente is not None and liquidez_corrente > 0:
        if liquidez_corrente > 1.5:
            pontos += 1; detalhes.append(f"+1 liquidez de curto prazo saudável ({liquidez_corrente:.2f})")
        elif liquidez_corrente < 1.0:
            pontos -= 1; detalhes.append(f"-1 liquidez de curto prazo apertada ({liquidez_corrente:.2f})")

    dy = div_info.get('dy_12m_real') if div_info else None
    if dy is not None and dy > 0.06:
        pontos += 1; detalhes.append(f"+1 yield efetivo (12m) atrativo ({dy*100:.1f}%)")

    return pontos, detalhes


def pontuar_acao_us(tec, fund, div_info):
    """Mesma lógica técnica de pontuar_acao, mas com limiares fundamentalistas
    próprios para o mercado americano — historicamente os múltiplos de P/L e P/VP
    do S&P 500 rodam mais altos que os do Ibovespa, então usar os mesmos limiares
    das ações brasileiras penalizaria injustamente boa parte das ações dos EUA."""
    pontos = 0
    detalhes = []

    if tec.get('sma200') is not None:
        if tec['preco_atual'] > tec['sma200']:
            pontos += 1; detalhes.append("+1 preço acima da MM200 (tendência de alta)")
        else:
            pontos -= 1; detalhes.append("-1 preço abaixo da MM200 (tendência de baixa)")

    if tec.get('rsi14') is not None:
        if tec['rsi14'] < 40:
            pontos += 1; detalhes.append("+1 RSI < 40 (não sobrecomprado)")
        elif tec['rsi14'] > 70:
            pontos -= 1; detalhes.append("-1 RSI > 70 (sobrecomprado)")

    _minima_periodo = tec['preco_atual'] / (1 + tec['dist_minima_pct'] / 100)
    _maxima_periodo = tec['preco_atual'] / (1 + tec['dist_maxima_pct'] / 100)
    if tec['dist_minima_pct'] < 15:
        pontos += 1
        _verbo_min = 'abaixo da' if tec['preco_atual'] < _minima_periodo else 'próximo da'
        detalhes.append(f"+1 {_verbo_min} mínima do período (atual {tec['preco_atual']:.2f}, mínima {_minima_periodo:.2f})")
    if tec['dist_maxima_pct'] > -5:
        pontos -= 1
        _verbo_max = 'acima da' if tec['preco_atual'] > _maxima_periodo else 'próximo da'
        detalhes.append(f"-1 {_verbo_max} máxima do período (atual {tec['preco_atual']:.2f}, máxima {_maxima_periodo:.2f})")

    pl = fund.get('pl')
    if pl is not None and pl > 0:
        if pl < 25:
            pontos += 1; detalhes.append(f"+1 P/L baixo p/ padrão americano ({pl:.1f})")
        elif pl > 40:
            pontos -= 1; detalhes.append(f"-1 P/L alto mesmo p/ padrão americano ({pl:.1f})")

    pvp = fund.get('pvp')
    if pvp is not None:
        if pvp < 4:
            pontos += 1; detalhes.append(f"+1 P/VP baixo p/ padrão americano ({pvp:.2f})")
        elif pvp > 10:
            pontos -= 1; detalhes.append(f"-1 P/VP alto mesmo p/ padrão americano ({pvp:.2f})")

    roe = fund.get('roe')
    if roe is not None and roe > 0.15:
        pontos += 1; detalhes.append(f"+1 ROE saudável ({roe*100:.1f}%)")

    dy = div_info.get('dy_12m_real') if div_info else None
    if dy is not None and dy > 0.02:
        pontos += 1; detalhes.append(f"+1 yield efetivo (12m) atrativo p/ padrão americano ({dy*100:.1f}%)")

    return pontos, detalhes


def pontuar_etf(tec, div_info, tipo):
    """ETFs não têm P/L, P/VP nem ROE (não são empresas, são cestas de ativos que
    seguem um índice) — então o critério aqui é deliberadamente mais simples que o
    de ações: só técnico (MM200, RSI, distância do período) + dividend yield, com
    limiar de yield próprio por mercado (B3 costuma distribuir mais que EUA)."""
    pontos = 0
    detalhes = []

    if tec.get('sma200') is not None:
        if tec['preco_atual'] > tec['sma200']:
            pontos += 1; detalhes.append("+1 preço acima da MM200 (tendência de alta)")
        else:
            pontos -= 1; detalhes.append("-1 preço abaixo da MM200 (tendência de baixa)")

    if tec.get('rsi14') is not None:
        if tec['rsi14'] < 40:
            pontos += 1; detalhes.append("+1 RSI < 40 (não sobrecomprado)")
        elif tec['rsi14'] > 70:
            pontos -= 1; detalhes.append("-1 RSI > 70 (sobrecomprado)")

    _minima_periodo = tec['preco_atual'] / (1 + tec['dist_minima_pct'] / 100)
    _maxima_periodo = tec['preco_atual'] / (1 + tec['dist_maxima_pct'] / 100)
    if tec['dist_minima_pct'] < 15:
        pontos += 1
        _verbo_min = 'abaixo da' if tec['preco_atual'] < _minima_periodo else 'próximo da'
        detalhes.append(f"+1 {_verbo_min} mínima do período (atual {tec['preco_atual']:.2f}, mínima {_minima_periodo:.2f})")
    if tec['dist_maxima_pct'] > -5:
        pontos -= 1
        _verbo_max = 'acima da' if tec['preco_atual'] > _maxima_periodo else 'próximo da'
        detalhes.append(f"-1 {_verbo_max} máxima do período (atual {tec['preco_atual']:.2f}, máxima {_maxima_periodo:.2f})")

    limite_dy = 0.04 if tipo == 'etf_br' else 0.015  # B3 costuma distribuir mais que ETFs dos EUA
    dy = div_info.get('dy_12m_real') if div_info else None
    if dy is not None and dy > limite_dy:
        pontos += 1; detalhes.append(f"+1 yield efetivo (12m) atrativo p/ ETF ({dy*100:.1f}%)")

    return pontos, detalhes


def pontuar_fii(tec, fund, div_info):
    pontos = 0
    detalhes = []

    if tec.get('sma200') is not None:
        if tec['preco_atual'] > tec['sma200']:
            pontos += 1; detalhes.append("+1 cota acima da MM200")
        else:
            pontos -= 1; detalhes.append("-1 cota abaixo da MM200")

    if tec.get('rsi14') is not None:
        if tec['rsi14'] < 40:
            pontos += 1; detalhes.append("+1 RSI < 40 (não sobrecomprado)")
        elif tec['rsi14'] > 70:
            pontos -= 1; detalhes.append("-1 RSI > 70 (sobrecomprado)")

    _minima_periodo = tec['preco_atual'] / (1 + tec['dist_minima_pct'] / 100)
    _maxima_periodo = tec['preco_atual'] / (1 + tec['dist_maxima_pct'] / 100)
    if tec['dist_minima_pct'] < 15:
        pontos += 1
        _verbo_min = 'abaixo da' if tec['preco_atual'] < _minima_periodo else 'próxima da'
        detalhes.append(f"+1 {_verbo_min} mínima do período (atual {tec['preco_atual']:.2f}, mínima {_minima_periodo:.2f})")
    if tec['dist_maxima_pct'] > -5:
        pontos -= 1
        _verbo_max = 'acima da' if tec['preco_atual'] > _maxima_periodo else 'próxima da'
        detalhes.append(f"-1 {_verbo_max} máxima do período (atual {tec['preco_atual']:.2f}, máxima {_maxima_periodo:.2f})")

    pvp = fund.get('pvp')
    if pvp is not None:
        if pvp < 0.95:
            pontos += 1; detalhes.append(f"+1 cota com desconto (P/VP {pvp:.2f})")
        elif pvp > 1.10:
            pontos -= 1; detalhes.append(f"-1 cota com ágio (P/VP {pvp:.2f})")

    # Indicadores operacionais exclusivos do Fundamentus (Yahoo Finance não
    # fornece dados de vacância/cap rate de FIIs). Não se aplica a FIIs "de
    # papel" (recebíveis/CRIs, sem imóveis físicos) — usa qtd_imoveis como
    # sinal: 0 (ou ausente) = FII de papel, pula essa parte da pontuação.
    # Caso real que motivou essa correção: RECR11.SA veio do Fundamentus com
    # vacância=100% e cap_rate=0% — valores de preenchimento do site pra
    # campos que não se aplicam a esse tipo de fundo, não um dado ruim de
    # verdade (ver Seção 4.2 do Relatório).
    segmento_fii = str(fund.get('segmento') or '').lower()
    segmento_indica_papel = any(
        p in segmento_fii for p in ('papel', 'recebív', 'recebiv', 'título', 'titulo', 'renda fixa'))
    qtd_imoveis_indica_papel = fund.get('qtd_imoveis') in (None, 0)
    vacancia_bruta_fii = fund.get('vacancia_media')
    vacancia_extrema_suspeita = vacancia_bruta_fii is not None and vacancia_bruta_fii >= 0.999
    eh_fii_de_tijolo = not (qtd_imoveis_indica_papel or segmento_indica_papel or vacancia_extrema_suspeita)

    vacancia = fund.get('vacancia_media')
    if eh_fii_de_tijolo and vacancia is not None:
        if vacancia < 0.05:
            pontos += 1; detalhes.append(f"+1 vacância baixa ({vacancia*100:.1f}%)")
        elif vacancia > 0.15:
            pontos -= 1; detalhes.append(f"-1 vacância alta ({vacancia*100:.1f}%)")

    cap_rate = fund.get('cap_rate')
    if eh_fii_de_tijolo and cap_rate is not None:
        if cap_rate > 0.08:
            pontos += 1; detalhes.append(f"+1 cap rate atrativo ({cap_rate*100:.1f}%)")
        elif cap_rate < 0.05:
            pontos -= 1; detalhes.append(f"-1 cap rate baixo ({cap_rate*100:.1f}%)")

    dy = div_info.get('dy_12m_real') if div_info else None
    if dy is not None and dy > 0.08:
        pontos += 1; detalhes.append(f"+1 yield efetivo (12m) atrativo ({dy*100:.1f}%)")

    return pontos, detalhes


def classificar(pontos):
    """Descreve o que os indicadores técnicos mostram — não é uma recomendação
    de compra/venda, é uma leitura do conjunto de sinais. Cabe ao usuário decidir
    o que fazer com essa informação."""
    if pontos >= 3:
        return "🟢 Indicadores técnicos majoritariamente favoráveis"
    elif pontos >= 1:
        return "🟡 Indicadores técnicos mistos, leve viés favorável"
    elif pontos >= -1:
        return "🟡 Indicadores técnicos mistos, sem direção clara"
    else:
        return "🔴 Indicadores técnicos majoritariamente desfavoráveis"


def avaliar_renda(div_info, tipo):
    """Avalia a QUALIDADE da renda passiva de um ativo, com base no histórico real
    de dividendos pagos — independente de o momento ser bom pra comprar ou vender.
    Pensado pra quem pretende manter o ativo e viver da renda, não do giro de preço."""
    if div_info is None:
        if tipo == 'acao_us':
            # Comum em ações de crescimento americanas (reinvestem lucro em vez de
            # distribuir) — tratar como "dados insuficientes" seria enganoso, já que
            # não é falha de dado, é a própria natureza do ativo.
            return 'NA', ["ativo não tem histórico de dividendos na fonte de dados — comum em "
                          "ações de crescimento que reinvestem lucro em vez de distribuir"]
        if tipo in ('etf_br', 'etf_us'):
            # Pode ser um ETF "de acumulação" (reinveste internamente em vez de
            # distribuir), um ETF de renda fixa cujo provento a fonte de dados não
            # capturou como "dividendo", ou simplesmente dado indisponível — o
            # notebook não distingue esses casos, por isso a mensagem é genérica.
            return 'NA', ["ETF não tem histórico de distribuição de proventos na fonte de dados — "
                          "pode ser um ETF de acumulação, um provento não capturado como 'dividendo' "
                          "pela fonte de dados (comum em ETFs de renda fixa), ou dado indisponível"]
        return None, ["sem histórico de dividendos suficiente na fonte de dados"]

    pontos = 0
    detalhes = []
    if tipo == 'fii':
        limite_dy = 0.08
    elif tipo == 'acao_us':
        limite_dy = 0.02  # yield médio do mercado americano é bem mais baixo que o brasileiro
    elif tipo == 'etf_br':
        limite_dy = 0.04
    elif tipo == 'etf_us':
        limite_dy = 0.015  # ETFs americanos (ex: SPY, QQQ) tipicamente distribuem pouco
    else:
        limite_dy = 0.06

    if div_info['dy_12m_real'] is not None:
        dy = div_info['dy_12m_real']
        if dy > limite_dy:
            pontos += 1; detalhes.append(f"+1 yield efetivo dos últimos 12m atrativo ({dy*100:.1f}%)")
        elif dy < limite_dy / 2:
            pontos -= 1; detalhes.append(f"-1 yield efetivo dos últimos 12m baixo ({dy*100:.1f}%)")

    if div_info['crescimento_yoy'] is not None:
        cresc = div_info['crescimento_yoy']
        if cresc > 0.05:
            pontos += 1; detalhes.append(f"+1 dividendos crescendo vs. ano anterior ({cresc*100:+.1f}%)")
        elif cresc < -0.15:
            pontos -= 1; detalhes.append(f"-1 dividendos caindo vs. ano anterior ({cresc*100:+.1f}%)")

    n_pag = div_info['n_pagamentos_12m']
    pagamentos_esperados = 10 if tipo == 'fii' else 2  # FIIs costumam pagar mensal, ações/ETFs costumam pagar 1-4x/ano
    if n_pag >= pagamentos_esperados:
        pontos += 1; detalhes.append(f"+1 pagamentos regulares ({n_pag} nos últimos 12m)")
    elif n_pag == 0:
        pontos -= 1; detalhes.append("-1 nenhum pagamento nos últimos 12 meses")

    return pontos, detalhes


def classificar_renda(pontos):
    if pontos is None:
        return "⚪ dados de dividendos insuficientes"
    elif pontos == 'NA':
        return "⚪ Não aplicável — foco em crescimento, não renda"
    elif pontos >= 2:
        return "🟢 renda saudável e crescente"
    elif pontos >= 0:
        return "🟡 renda estável"
    else:
        return "🔴 atenção: renda em deterioração"


def leitura_combinada(pontos_timing, pontos_renda):
    """Combina o sinal técnico com a qualidade da renda numa LEITURA descritiva do
    conjunto de indicadores — não é uma recomendação de compra/venda/manutenção;
    descreve o que os números mostram, cabendo ao usuário decidir o que fazer com
    essa informação. (Renomeada de sugerir_estrategia(): o nome antigo sugeria uma
    ação; o conteúdo sempre foi, e continua sendo, só leitura de dado.)"""
    if pontos_renda == 'NA':
        pontos_renda = None  # "não aplicável" é neutro pra fins desta leitura, não penaliza

    if pontos_renda is not None and pontos_renda < 0:
        return "🔴 Renda em deterioração — indicador mais relevante que o preço neste caso"

    if pontos_timing <= -2:
        if pontos_renda is None or pontos_renda >= 0:
            return "🔵 Preço no percentil superior do período, com renda estável"
        return "🔴 Preço no percentil superior do período, sem histórico de renda confiável"

    if pontos_timing >= 3:
        return "🟢 Indicadores técnicos e de renda apontam na mesma direção favorável"

    if pontos_renda is not None and pontos_renda >= 2:
        return "🟡 Renda saudável, sem sinal técnico forte no momento"

    return "🟡 Sem direção clara nos indicadores"

print("Motor de pontuação pronto. Limites são ajustáveis nesta célula.")
print("Agora com três saídas descritivas: leitura técnica, qualidade da renda, e leitura combinada das duas.")
print("Suporta 5 tipos de ativo: acao (B3), fii (B3), acao_us (ações EUA), etf_br e etf_us (com limiares próprios).")


In [ ]:
#@title 🚀 7. Rodar a análise da carteira inteira

botao_analisar = widgets.Button(description="🚀 Analisar carteira", button_style='primary', icon='play')
saida_analise = widgets.Output()

def parse_tickers(texto):
    """Quebra a lista de tickers por vírgula. Corrige automaticamente um erro de
    digitação comum: vírgula no lugar do ponto antes do sufixo de bolsa
    (ex: 'VALE3, SA' vira, sem querer, dois tokens 'VALE3' e 'SA' — aqui a gente
    detecta esse padrão e reconstrói como 'VALE3.SA')."""
    brutos = [t.strip().upper() for t in texto.split(',') if t.strip()]

    tickers = []
    i = 0
    while i < len(brutos):
        atual = brutos[i]
        proximo = brutos[i + 1] if i + 1 < len(brutos) else None
        if proximo in ('SA', 'US') and not atual.endswith('.' + proximo):
            tickers.append(f"{atual}.{proximo}")
            i += 2
        else:
            tickers.append(atual)
            i += 1
    return tickers

def analisar_grupo(tickers, tipo, periodo):
    linhas = []
    detalhes_por_ativo = {}
    moeda = 'US$' if tipo in ('acao_us', 'etf_us') else 'R$'
    for ticker in tickers:
        tec = indicadores_tecnicos(ticker, periodo)
        if tec is None:
            linhas.append({'Ticker': ticker, 'Sinal de timing': '⚠️ sem dados suficientes', 'Pontos': None})
            continue

        div_info = historico_dividendos(ticker, tec['preco_atual'])

        if tipo == 'fii':
            fund = indicadores_fundamentalistas_fii(ticker)
            pontos, detalhes = pontuar_fii(tec, fund, div_info)
        elif tipo == 'acao_us':
            fund = indicadores_fundamentalistas_acao(ticker)  # mesmos campos do Yahoo, funciona p/ EUA também
            pontos, detalhes = pontuar_acao_us(tec, fund, div_info)
        elif tipo in ('etf_br', 'etf_us'):
            fund = indicadores_fundamentalistas_acao(ticker)  # geralmente vazio p/ ETF, mas não quebra nada
            pontos, detalhes = pontuar_etf(tec, div_info, tipo)
        else:
            fund = indicadores_fundamentalistas_acao(ticker)
            pontos, detalhes = pontuar_acao(tec, fund, div_info)

        pontos_renda, detalhes_renda = avaliar_renda(div_info, tipo)

        # Pregão da B3 só se aplica a ativos negociados na B3 (fuso e horário de
        # negociação diferentes para ativos americanos).
        pregao_aberto = pregao_provavelmente_aberto(tec['data_ultima_cotacao']) if tipo in ('acao', 'fii', 'etf_br') else False

        if tec.get('n_dias_truncados'):
            detalhes.append(
                f"⚠️ {tec['n_dias_truncados']} dia(s) mais antigos do histórico foram descartados "
                f"(preço numa escala muito diferente da atual — possível evento societário não "
                f"ajustado, ex: agrupamento de cotas). Indicadores calculados com os "
                f"{tec['dias_usados']} dias restantes, mais confiáveis."
            )
        elif tec.get('n_outliers_corrigidos'):
            detalhes.append(f"⚠️ {tec['n_outliers_corrigidos']} ponto(s) de preço suspeito(s) corrigido(s) no histórico")

        if pregao_aberto:
            detalhes.append(
                "📊 Pregão provavelmente em andamento — este preço é o mais recente disponível, "
                "não necessariamente o fechamento definitivo do dia. Os indicadores podem mudar "
                "se a análise for rodada de novo após o fechamento (~18h, horário de Brasília)."
            )

        cotacao_texto = tec['data_ultima_cotacao'].strftime('%d/%m/%Y')
        if pregao_aberto:
            cotacao_texto += ' (pregão aberto)'

        linhas.append({
            'Ticker': ticker + (' ⚠️' if (tec.get('n_outliers_corrigidos') or tec.get('n_dias_truncados')) else ''),
            'Preço': f"{moeda} {tec['preco_atual']:.2f}",
            'Última cotação': cotacao_texto,
            'RSI(14)': round(tec['rsi14'], 1) if tec['rsi14'] else None,
            'Dist. mín. (p5)': f"{tec['dist_minima_pct']:.1f}%",
            'Dist. máx. (p95)': f"{tec['dist_maxima_pct']:.1f}%",
            'P/VP': round(fund.get('pvp'), 2) if fund.get('pvp') else 'N/D',
            'Yield 12m (real)': f"{div_info['dy_12m_real']*100:.1f}%" if div_info and div_info['dy_12m_real'] is not None else 'N/D',
            'Dividendos 12m': f"{moeda} {div_info['soma_12m']:.2f}" if div_info else 'N/D',
            'Cresc. dividendos (a/a)': f"{div_info['crescimento_yoy']*100:+.1f}%" if div_info and div_info['crescimento_yoy'] is not None else 'N/D',
            'Nº pagamentos 12m': div_info['n_pagamentos_12m'] if div_info else 'N/D',
            'Sinal de timing': classificar(pontos),
            'Qualidade da renda': classificar_renda(pontos_renda),
            'Leitura combinada': leitura_combinada(pontos, pontos_renda),
        })
        detalhes_por_ativo[ticker] = {'timing': detalhes, 'renda': detalhes_renda}

    return pd.DataFrame(linhas), detalhes_por_ativo

def analisar_carteira(b):
    with saida_analise:
        clear_output()
        tickers_acoes = parse_tickers(caixa_acoes.value)
        tickers_fiis = parse_tickers(caixa_fiis.value)
        tickers_acoes_us = parse_tickers(caixa_acoes_us.value)
        tickers_etf_br = parse_tickers(caixa_etf_br.value)
        tickers_etf_us = parse_tickers(caixa_etf_us.value)
        periodo = periodo_historico.value

        agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%d/%m/%Y às %H:%M (horário de Brasília)')
        print(f"🕒 Análise executada em {agora}")
        print(f"Analisando {len(tickers_acoes)} ações BR, {len(tickers_fiis)} FIIs, {len(tickers_acoes_us)} ações EUA, "
              f"{len(tickers_etf_br)} ETFs BR e {len(tickers_etf_us)} ETFs EUA...\n")

        def imprimir_detalhes(det):
            for ticker, grupos in det.items():
                print(f"\n  {ticker}:")
                if grupos.get('timing'):
                    print("    Timing (compra/venda):")
                    for d in grupos['timing']:
                        print(f"      {d}")
                if grupos.get('renda'):
                    print("    Qualidade da renda:")
                    for d in grupos['renda']:
                        print(f"      {d}")

        if tickers_acoes:
            df_acoes, det_acoes = analisar_grupo(tickers_acoes, 'acao', periodo)
            print("📈 AÇÕES (B3)")
            display(df_acoes)
            imprimir_detalhes(det_acoes)
            print("\n" + "-"*70 + "\n")

        if tickers_fiis:
            df_fiis, det_fiis = analisar_grupo(tickers_fiis, 'fii', periodo)
            print("🏢 FIIs")
            display(df_fiis)
            imprimir_detalhes(det_fiis)
            print("\n" + "-"*70 + "\n")

        if tickers_acoes_us:
            df_acoes_us, det_acoes_us = analisar_grupo(tickers_acoes_us, 'acao_us', periodo)
            print("🇺🇸 AÇÕES (EUA)")
            display(df_acoes_us)
            imprimir_detalhes(det_acoes_us)
            print("\n" + "-"*70 + "\n")

        if tickers_etf_br:
            df_etf_br, det_etf_br = analisar_grupo(tickers_etf_br, 'etf_br', periodo)
            print("📊 ETFs (B3)")
            display(df_etf_br)
            imprimir_detalhes(det_etf_br)
            print("\n" + "-"*70 + "\n")

        if tickers_etf_us:
            df_etf_us, det_etf_us = analisar_grupo(tickers_etf_us, 'etf_us', periodo)
            print("📊 ETFs (EUA)")
            display(df_etf_us)
            imprimir_detalhes(det_etf_us)

        print("\n📌 Como ler esta tabela: 'Sinal de timing' descreve a tendência predominante dos")
        print("   indicadores técnicos no momento (não é uma recomendação de compra/venda).")
        print("   'Qualidade da renda' descreve se os dividendos pagos têm sido crescentes e")
        print("   consistentes. 'Leitura combinada' junta as duas descrições num só resumo —")
        print("   continua sendo descrição do que os números mostram, não uma indicação do que")
        print("   fazer. A decisão de comprar, manter ou vender é sempre do usuário.")
        print("   ⚠️ ao lado de um ticker NÃO é um sinal — é só aviso de que o histórico de preço")
        print("   desse ativo precisou de algum ajuste (dado ruim do provedor), sem relação com os")
        print("   sinais coloridos. Detalhes de cada ajuste aparecem no detalhamento abaixo.")
        print("   ⚪ 'Não aplicável' na qualidade da renda (comum em ações EUA de crescimento e em")
        print("   ETFs de acumulação) não é um sinal negativo — só indica que o ativo não distribui")
        print("   proventos regularmente. ETFs não têm P/L/P/VP/ROE (não são empresas), por isso o")
        print("   componente técnico deles usa só tendência/RSI/distância do período mais yield.")
        print("   Esta ferramenta é estritamente informativa: não constitui recomendação de")
        print("   investimento, consultoria ou análise de valores mobiliários.")

botao_analisar.on_click(analisar_carteira)
display(botao_analisar, saida_analise)


## 🔍 8. Analisar um ativo específico

Quer olhar só um ativo, com todos os critérios (timing, qualidade da renda, estratégia sugerida)
e o gráfico de preço/médias móveis/RSI, sem rodar a carteira inteira de novo? Use a célula abaixo.
Ela usa exatamente a mesma lógica e os mesmos dados da Seção 7 — os números batem 100% com a tabela
principal, é só uma forma mais direta de olhar um ativo isolado.


In [ ]:
#@title 🔍 8. Analisar um ativo específico

caixa_ticker_individual = widgets.Text(
    value='PETR4.SA',
    description='Ticker:',
    style={'description_width': 'initial'}
)
tipo_ticker_individual = widgets.Dropdown(
    options=[('Ação (B3)', 'acao'), ('FII', 'fii'), ('Ação (EUA)', 'acao_us'),
             ('ETF (B3)', 'etf_br'), ('ETF (EUA)', 'etf_us')],
    value='acao',
    description='Tipo:',
    style={'description_width': 'initial'}
)
botao_individual = widgets.Button(description="🔍 Analisar ativo", button_style='info', icon='search')
saida_individual = widgets.Output()

def analisar_ativo_individual(b):
    with saida_individual:
        clear_output()
        ticker = caixa_ticker_individual.value.strip().upper()
        tipo = tipo_ticker_individual.value
        periodo = periodo_historico.value

        # Reaproveita a MESMA função usada na análise da carteira inteira (Seção 7),
        # só que com uma lista de 1 ticker — garante que os números batem exatamente
        # com os da tabela principal, sem duplicar lógica de cálculo em dois lugares.
        agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%d/%m/%Y às %H:%M (horário de Brasília)')
        print(f"🕒 Análise executada em {agora}")
        df, detalhes_por_ativo = analisar_grupo([ticker], tipo, periodo)

        if df.empty or 'Sinal de timing' not in df.columns or df.iloc[0].get('Sinal de timing') == '⚠️ sem dados suficientes':
            print(f"⚠️ Não foi possível obter dados suficientes para {ticker}.")
            return

        linha = df.iloc[0]
        nomes_tipo = {'acao': 'Ação (B3)', 'fii': 'FII', 'acao_us': 'Ação (EUA)',
                      'etf_br': 'ETF (B3)', 'etf_us': 'ETF (EUA)'}
        print(f"📌 {ticker}  —  {nomes_tipo.get(tipo, tipo)}\n")
        for coluna in df.columns:
            if coluna == 'Ticker':
                continue
            print(f"  {coluna}: {linha[coluna]}")

        print()
        grupos = detalhes_por_ativo.get(ticker, {})
        if grupos.get('timing'):
            print("  Timing (compra/venda):")
            for d in grupos['timing']:
                print(f"    {d}")
        if grupos.get('renda'):
            print("  Qualidade da renda:")
            for d in grupos['renda']:
                print(f"    {d}")

        tec = indicadores_tecnicos(ticker, periodo)
        precos = tec['historico']
        mm50 = precos.rolling(50).mean()
        mm200 = precos.rolling(200).mean()
        rsi = calcular_rsi(precos, 14)

        # Diagnóstico automático: mostra os preços mais extremos do histórico BRUTO
        # (antes de qualquer limpeza), pra você conferir rapidamente se uma queda ou
        # alta forte no gráfico é dado real ou coisa de provedor de dados. Reaproveita
        # o cache que indicadores_tecnicos() já preencheu logo acima — evita baixar o
        # mesmo dado duas vezes no mesmo clique.
        fechamento_bruto = _cache_get(f"precos:{ticker}:{periodo}")
        if fechamento_bruto is None:
            hist_bruto = yf.download(ticker, period=periodo, progress=False)
            fechamento_bruto = hist_bruto['Close']
            if isinstance(fechamento_bruto, pd.DataFrame):
                fechamento_bruto = fechamento_bruto.squeeze(axis=1)
            fechamento_bruto = fechamento_bruto.dropna()

        print("\n🔍 Diagnóstico do histórico bruto (antes de qualquer limpeza):")
        print(f"  Total de dias no histórico: {len(fechamento_bruto)}")
        print(f"  Preço atual: {fechamento_bruto.iloc[-1]:.2f}")
        print("\n  10 MENORES preços do período (data e valor):")
        print(fechamento_bruto.nsmallest(10))
        print("\n  10 MAIORES preços do período (data e valor):")
        print(fechamento_bruto.nlargest(10))
        print("\n  Se algum desses valores parecer fora do padrão dos vizinhos, é sinal de dado ruim")
        print("  do provedor (o que o notebook já tenta corrigir automaticamente); se os valores")
        print("  parecerem consistentes com uma queda/alta real e gradual, é movimento genuíno do ativo.")

        # Leitura automática do padrão dos preços extremos: só faz sentido analisar ciclos
        # do lado (mínima ou máxima) que o preço ATUAL está de fato perto — analisar o lado
        # errado (ex: procurar padrão na mínima quando o preço está no meio do range) gera
        # leitura confusa/alarmista à toa.
        data_mais_recente = fechamento_bruto.index.max()
        ciclos_minimas = agrupar_datas_em_ciclos(list(fechamento_bruto.nsmallest(10).index))
        ciclos_maximas = agrupar_datas_em_ciclos(list(fechamento_bruto.nlargest(10).index))
        dias_desde_ciclo_recente_min = (data_mais_recente - ciclos_minimas[-1][-1]).days
        dias_desde_ciclo_recente_max = (data_mais_recente - ciclos_maximas[-1][-1]).days

        perto_da_minima = tec['dist_minima_pct'] < 15
        perto_da_maxima = tec['dist_maxima_pct'] > -5

        print("\n🧭 Leitura automática:")
        if perto_da_minima:
            if len(ciclos_minimas) == 1 and dias_desde_ciclo_recente_min <= 90:
                print("  O preço está perto da mínima do período, e os menores preços do histórico estão")
                print("  concentrados nos últimos ~3 meses — queda recente e aparentemente real, sem")
                print("  precedente parecido dentro do período analisado.")
            elif len(ciclos_minimas) > 1:
                primeiro = ciclos_minimas[0]
                ultimo = ciclos_minimas[-1]
                print(f"  O preço está perto da mínima do período, e o ativo já esteve num patamar parecido")
                print(f"  antes: identifiquei {len(ciclos_minimas)} período(s) de mínima separados por")
                print(f"  recuperações no meio. O mais antigo foi por volta de {primeiro[0].date()}, e o mais")
                print(f"  recente começou por volta de {ultimo[0].date()}. Ou seja, essa não é a primeira vez")
                print("  que o ativo chega perto desse preço — já recuperou dessa faixa ao menos uma vez")
                print("  antes. Vale entender se o motivo daquela recuperação anterior (resultado, notícia,")
                print("  ciclo do setor) ainda se aplica agora, ou se dessa vez é diferente.")
            else:
                print("  O preço está perto da mínima do período, mas os menores preços do histórico não")
                print("  seguem um padrão simples de queda contínua nem de ciclos repetidos claros. Vale")
                print("  conferir as datas acima manualmente.")
        elif perto_da_maxima:
            if len(ciclos_maximas) == 1 and dias_desde_ciclo_recente_max <= 90:
                print("  O preço está perto da máxima do período, e os maiores preços do histórico estão")
                print("  concentrados nos últimos ~3 meses — alta recente e aparentemente real, sem")
                print("  precedente parecido dentro do período analisado.")
            elif len(ciclos_maximas) > 1:
                primeiro = ciclos_maximas[0]
                ultimo = ciclos_maximas[-1]
                print(f"  O preço está perto da máxima do período, e o ativo já chegou perto desse patamar")
                print(f"  alto antes: identifiquei {len(ciclos_maximas)} período(s) de máxima separados por")
                print(f"  quedas no meio. O mais antigo foi por volta de {primeiro[0].date()}, e o mais recente")
                print(f"  começou por volta de {ultimo[0].date()}. Ou seja, o ativo já esbarrou nesse teto antes")
                print("  e recuou depois — vale considerar se esse nível costuma ser um ponto de resistência")
                print("  pra esse ativo.")
            else:
                print("  O preço está perto da máxima do período, mas os maiores preços do histórico não")
                print("  seguem um padrão simples de alta contínua nem de ciclos repetidos claros. Vale")
                print("  conferir as datas acima manualmente.")
        else:
            data_minima_abs = fechamento_bruto.idxmin()
            valor_minima_abs = fechamento_bruto.min()
            data_maxima_abs = fechamento_bruto.idxmax()
            valor_maxima_abs = fechamento_bruto.max()
            moeda_leitura = 'US$' if tipo in ('acao_us', 'etf_us') else 'R$'
            print(f"  O preço atual ({moeda_leitura} {tec['preco_atual']:.2f}) não está perto nem da mínima nem da máxima")
            print(f"  do período — está numa faixa intermediária. A mínima do período foi {moeda_leitura}")
            print(f"  {valor_minima_abs:.2f} em {data_minima_abs.date()}, e a máxima foi {moeda_leitura} {valor_maxima_abs:.2f}")
            print(f"  em {data_maxima_abs.date()}. Nada de especial pra verificar aqui.")

        if tec['dist_minima_pct'] < 15 and tec.get('rsi14') is not None and tec['rsi14'] < 35:
            print("\n  ⚠️ Atenção: o ativo está próximo da mínima do período E com RSI baixo ao mesmo")
            print("  tempo. Isso pode ser uma barganha genuína (preço descontado, prestes a reagir),")
            print("  ou pode ser uma queda estrutural que ainda não acabou — RSI baixo não garante que")
            print("  o fundo já foi atingido. Este notebook não distingue as duas coisas; vale checar")
            print("  se há notícias recentes (resultados, dívida, setor) que expliquem a queda antes")
            print("  de decidir.")

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                        gridspec_kw={'height_ratios': [3, 1]})
        ax1.plot(precos.index, precos.values, label='Preço', color='#1f77b4')
        ax1.plot(mm50.index, mm50.values, label='MM50', color='#ff7f0e', linewidth=1)
        ax1.plot(mm200.index, mm200.values, label='MM200', color='#d62728', linewidth=1)
        titulo_grafico = f"{ticker} — preço e médias móveis"
        if tec.get('n_dias_truncados'):
            titulo_grafico += f" (histórico limpo: {tec['dias_usados']} dias confiáveis)"
        ax1.set_title(titulo_grafico)
        ax1.legend(); ax1.grid(alpha=0.3)

        ax2.plot(rsi.index, rsi.values, color='#9467bd')
        ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
        ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
        ax2.set_title("RSI (14)"); ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

botao_individual.on_click(analisar_ativo_individual)
display(widgets.VBox([caixa_ticker_individual, tipo_ticker_individual, botao_individual, saida_individual]))


## 💹 9. Ganhos e proventos da posição

Duas formas de acompanhar o ganho de um ativo, conforme o quanto de detalhe você tiver à mão:

- **9.1 Ganho de posição aberta**: só quantidade, preço médio e data de início — não exige nenhum histórico de operações.
- **9.2 Registro de operações**: se você quiser registrar compras e vendas ao longo do tempo, a partir de agora, a ferramenta calcula o ganho já realizado nas vendas, o que ainda está em aberto, e a renda recebida — sem precisar reconstruir operações antigas.

In [ ]:
#@title 💹 9. Ganhos e proventos da posição

def _dividendos_brutos_cache(ticker):
    """Busca (com cache) o histórico bruto de dividendos de um ticker — mesma
    fonte usada por historico_dividendos() na Seção 5, reaproveitada aqui pra
    somar proventos recebidos num período qualquer."""
    chave_cache = f"dividendos:{ticker}"
    divs = _cache_get(chave_cache)
    if divs is None:
        try:
            divs = yf.Ticker(ticker).dividends
        except Exception:
            return pd.Series(dtype=float)
        if divs is None or divs.empty:
            return pd.Series(dtype=float)
        _cache_set(chave_cache, divs)
    if divs.index.tz is not None:
        divs = divs.copy()
        divs.index = divs.index.tz_localize(None)
    return divs


def calcular_ganho_posicao_aberta(quantidade, preco_medio, data_inicio, preco_atual, dividendos=None):
    """Ganho de uma posição ainda aberta, a partir de dado agregado mínimo
    (quantidade, preço médio, data de início) — não exige nenhum histórico de
    operações. A renda recebida é somada automaticamente a partir do
    histórico real de dividendos do ativo desde a data de início informada."""
    if quantidade <= 0:
        raise ValueError("quantidade deve ser positiva")
    if preco_medio <= 0:
        raise ValueError("preco_medio deve ser positivo")

    valor_investido = quantidade * preco_medio
    valor_atual = quantidade * preco_atual
    ganho_preco = valor_atual - valor_investido
    ganho_preco_pct = ganho_preco / valor_investido

    renda_recebida = 0.0
    if dividendos is not None and not dividendos.empty:
        corte = pd.Timestamp(data_inicio)
        renda_recebida = float(dividendos[dividendos.index >= corte].sum()) * quantidade

    ganho_total = ganho_preco + renda_recebida
    ganho_total_pct = ganho_total / valor_investido

    return {
        'valor_investido': valor_investido, 'valor_atual': valor_atual,
        'ganho_preco': ganho_preco, 'ganho_preco_pct': ganho_preco_pct,
        'renda_recebida': renda_recebida,
        'ganho_total': ganho_total, 'ganho_total_pct': ganho_total_pct,
    }


def avaliar_operacoes(operacoes, preco_atual=None, dividendos=None):
    """Processa uma lista de operações de compra/venda (dicts com 'data',
    'tipo', 'quantidade', 'preco') em ordem cronológica, usando custo médio
    ponderado — o mesmo método usado pela Receita Federal para apuração de
    ganho de capital no Brasil. Registro incremental e opcional: não exige
    reconstrução do histórico anterior ao uso desta funcionalidade."""
    if not operacoes:
        raise ValueError("é necessário informar ao menos uma operação")

    ops = sorted(operacoes, key=lambda o: o['data'])
    quantidade_aberta = 0.0
    custo_medio = 0.0
    ganho_realizado = 0.0
    detalhes = []

    for op in ops:
        if op['tipo'] == 'compra':
            novo_custo_total = custo_medio * quantidade_aberta + op['preco'] * op['quantidade']
            quantidade_aberta += op['quantidade']
            custo_medio = novo_custo_total / quantidade_aberta if quantidade_aberta > 0 else 0.0
            detalhes.append(f"{op['data']}: compra de {op['quantidade']:.0f} a {op['preco']:.2f} "
                             f"— posição passa a {quantidade_aberta:.0f} (custo médio {custo_medio:.2f})")
        elif op['tipo'] == 'venda':
            if op['quantidade'] > quantidade_aberta + 1e-9:
                raise ValueError(f"venda de {op['quantidade']} em {op['data']} excede a posição em "
                                  f"aberto ({quantidade_aberta}) — confira as operações informadas")
            ganho_da_venda = (op['preco'] - custo_medio) * op['quantidade']
            ganho_realizado += ganho_da_venda
            quantidade_aberta -= op['quantidade']
            detalhes.append(f"{op['data']}: venda de {op['quantidade']:.0f} a {op['preco']:.2f} "
                             f"(custo médio {custo_medio:.2f}) — ganho realizado de {ganho_da_venda:+.2f}")
            if quantidade_aberta <= 1e-9:
                quantidade_aberta = 0.0
                custo_medio = 0.0
        else:
            raise ValueError(f"tipo de operação inválido: {op['tipo']!r} (use 'compra' ou 'venda')")

    ganho_nao_realizado = None
    if quantidade_aberta > 0 and preco_atual is not None:
        ganho_nao_realizado = (preco_atual - custo_medio) * quantidade_aberta

    renda_recebida = 0.0
    if dividendos is not None and not dividendos.empty:
        data_inicio = ops[0]['data']
        corte = pd.Timestamp(data_inicio)
        soma_por_cota = float(dividendos[dividendos.index >= corte].sum())
        qtd_acum, quantidades = 0.0, []
        for op in ops:
            qtd_acum += op['quantidade'] if op['tipo'] == 'compra' else -op['quantidade']
            quantidades.append(max(qtd_acum, 0.0))
        quantidade_media_periodo = sum(quantidades) / len(quantidades) if quantidades else 0.0
        renda_recebida = soma_por_cota * quantidade_media_periodo

    ganho_total = ganho_realizado + (ganho_nao_realizado or 0.0) + renda_recebida

    return {
        'quantidade_aberta': quantidade_aberta,
        'preco_medio_aberto': custo_medio if quantidade_aberta > 0 else None,
        'ganho_realizado': ganho_realizado, 'ganho_nao_realizado': ganho_nao_realizado,
        'renda_recebida': renda_recebida, 'ganho_total': ganho_total, 'detalhes': detalhes,
    }


# --- Painel 9.1: Ganho de posição aberta (dado agregado, sem histórico) ---
ticker_ganho = widgets.Text(value='PETR4.SA', description='Ticker:', style={'description_width': 'initial'})
qtd_ganho = widgets.FloatText(value=100, description='Quantidade:', style={'description_width': 'initial'})
preco_medio_ganho = widgets.FloatText(value=30.0, description='Preço médio:', style={'description_width': 'initial'})
data_inicio_ganho = widgets.Text(value='2024-01-01', description='Data início (AAAA-MM-DD):', style={'description_width': 'initial'})
botao_ganho = widgets.Button(description="💹 Calcular ganho", button_style='success', icon='calculator')
saida_ganho = widgets.Output()

def _calcular_ganho_click(b):
    with saida_ganho:
        clear_output()
        ticker = ticker_ganho.value.strip().upper()
        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        if tec is None:
            print(f"⚠️ Não foi possível obter dados suficientes para {ticker}.")
            return
        divs = _dividendos_brutos_cache(ticker)
        r = calcular_ganho_posicao_aberta(
            quantidade=qtd_ganho.value, preco_medio=preco_medio_ganho.value,
            data_inicio=data_inicio_ganho.value, preco_atual=tec['preco_atual'], dividendos=divs,
        )
        print(f"📌 {ticker} — {qtd_ganho.value:.0f} cotas desde {data_inicio_ganho.value}\n")
        print(f"  Valor investido: R$ {r['valor_investido']:.2f}")
        print(f"  Valor atual: R$ {r['valor_atual']:.2f}")
        print(f"  Ganho de preço: R$ {r['ganho_preco']:+.2f} ({r['ganho_preco_pct']*100:+.1f}%)")
        print(f"  Renda recebida (proventos desde o início): R$ {r['renda_recebida']:.2f}")
        print(f"  Ganho total: R$ {r['ganho_total']:+.2f} ({r['ganho_total_pct']*100:+.1f}%)")

botao_ganho.on_click(_calcular_ganho_click)
display(widgets.HTML("<h3>9.1 Ganho de posição aberta (dado agregado)</h3>"))
display(widgets.VBox([ticker_ganho, qtd_ganho, preco_medio_ganho, data_inicio_ganho, botao_ganho, saida_ganho]))


# --- Painel 9.2: Registro de operações (compra/venda) e posições encerradas ---
ticker_ops = widgets.Text(value='PETR4.SA', description='Ticker:', style={'description_width': 'initial'})
texto_ops = widgets.Textarea(
    value='2023-05-10,compra,100,25.00\n2023-11-20,compra,100,30.00\n2024-08-05,venda,80,35.00',
    description='Operações:', placeholder='data,tipo,quantidade,preco (uma por linha)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px', height='90px'),
)
botao_ops = widgets.Button(description="📋 Avaliar operações", button_style='success', icon='list')
saida_ops = widgets.Output()

def _avaliar_operacoes_click(b):
    with saida_ops:
        clear_output()
        ticker = ticker_ops.value.strip().upper()
        linhas = [l.strip() for l in texto_ops.value.split('\n') if l.strip()]
        operacoes = []
        for linha in linhas:
            try:
                data_str, tipo, qtd_str, preco_str = [p.strip() for p in linha.split(',')]
                operacoes.append({
                    'data': pd.Timestamp(data_str), 'tipo': tipo,
                    'quantidade': float(qtd_str), 'preco': float(preco_str),
                })
            except Exception:
                print(f"⚠️ Linha inválida, ignorada: {linha!r} (formato esperado: data,tipo,quantidade,preco)")

        if not operacoes:
            print("⚠️ Nenhuma operação válida informada.")
            return

        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        preco_atual = tec['preco_atual'] if tec is not None else None
        divs = _dividendos_brutos_cache(ticker)

        try:
            r = avaliar_operacoes(operacoes, preco_atual=preco_atual, dividendos=divs)
        except ValueError as e:
            print(f"⚠️ {e}")
            return

        print(f"📌 {ticker} — {len(operacoes)} operação(ões) registrada(s)\n")
        for d in r['detalhes']:
            print(" ", d)
        print()
        if r['quantidade_aberta'] > 0:
            print(f"  Posição ainda aberta: {r['quantidade_aberta']:.0f} cotas a custo médio R$ {r['preco_medio_aberto']:.2f}")
            if r['ganho_nao_realizado'] is not None:
                print(f"  Ganho não realizado: R$ {r['ganho_nao_realizado']:+.2f}")
        else:
            print("  Posição totalmente encerrada.")
        print(f"  Ganho realizado (vendas já feitas): R$ {r['ganho_realizado']:+.2f}")
        print(f"  Renda recebida no período: R$ {r['renda_recebida']:.2f}")
        print(f"  Ganho total: R$ {r['ganho_total']:+.2f}")
        print("\n  📌 A renda recebida é aproximada pela quantidade média mantida ao longo do")
        print("  período coberto pelas operações informadas — não célula a célula por operação.")

botao_ops.on_click(_avaliar_operacoes_click)
display(widgets.HTML("<h3>9.2 Registro de operações e posições encerradas</h3>"))
display(widgets.HTML("<p>Uma operação por linha, no formato <code>data,tipo,quantidade,preco</code> "
                      "(ex: <code>2024-01-15,compra,100,25.50</code>). Não é preciso reconstruir "
                      "operações antigas — comece a registrar a partir de agora.</p>"))
display(widgets.VBox([ticker_ops, texto_ops, botao_ops, saida_ops]))


## 💼 9.3 Ganho consolidado da carteira (via planilha)

Importa a carteira de um arquivo Excel/CSV ou Google Sheets, combinando duas abas por ativo:

- **Resumo** — a posição inicial de cada ativo, o ponto de partida: `ticker`, `quantidade`, `preco_medio` (ou `valor_investido`), `data_inicio`.
- **Operações** — as movimentações feitas a partir dali: `ticker`, `tipo` (compra/venda), `quantidade`, `preco`, `data`.

Um ativo pode ter só Resumo, só Operações, ou as duas — quando tiver as duas, a posição inicial (Resumo) entra como o primeiro registro, e cada operação lançada depois é aplicada **em cima** dela, ajustando quantidade, preço médio e ganho automaticamente. Não existe "sincronizar manualmente" — a cada execução, tudo é recalculado do zero a partir do que estiver nas duas abas naquele momento.

In [ ]:
#@title 💼 9.3 Ganho consolidado da carteira (via planilha)

import re

_PADRAO_TICKER_B3 = re.compile(r'^[A-Z]{4}\d{1,2}$')

def _normalizar_ticker(ticker_bruto):
    """Limpa espaços e, se o ticker parecer um código B3 (4 letras + 1-2
    dígitos) sem o sufixo '.SA', completa automaticamente — ex: 'POMO4' ->
    'POMO4.SA'. Devolve (ticker_normalizado, foi_corrigido: bool)."""
    t = str(ticker_bruto).strip().upper()
    if t.endswith('.SA') or '.' in t or '-' in t:
        return t, False
    if _PADRAO_TICKER_B3.match(t):
        return t + '.SA', True
    return t, False


# ============================================================================
# MODELO DE DADOS: a aba "Resumo" é o PONTO DE PARTIDA (posição inicial de
# cada ativo — o "saldo de abertura"); a aba "Operações" traz as
# MOVIMENTAÇÕES feitas a partir dali. As duas são combinadas por ativo:
# a posição inicial (se existir) entra como a primeira "compra" sintética,
# e cada operação lançada depois se soma em cima dela — igual um extrato.
# Um ativo pode ter só Resumo, só Operações, ou as duas (mais comum).
# ============================================================================

def _normalizar_aba_resumo(df):
    """Colunas: ticker, quantidade, data_inicio, e preco_medio OU
    valor_investido (pelo menos um dos dois — se só valor_investido vier
    preenchido, o preço médio é calculado como valor_investido/quantidade)."""
    mapa = {}
    for col in df.columns:
        c = str(col).strip().lower()
        c = (c.replace('í', 'i').replace('é', 'e').replace('ó', 'o')
               .replace('ã', 'a').replace('ç', 'c').replace('_', ' '))
        if c in ('ticker', 'ativo', 'codigo', 'papel'):
            mapa[col] = 'ticker'
        elif c in ('quantidade', 'qtd', 'qtde', 'quantity'):
            mapa[col] = 'quantidade'
        elif c in ('preco medio', 'pm', 'preco medio r$', 'preco', 'preco entrada', 'preco compra'):
            mapa[col] = 'preco_medio'
        elif c in ('valor investido', 'valor total', 'valor aplicado', 'total investido'):
            mapa[col] = 'valor_investido'
        elif c in ('data inicio', 'data', 'data compra', 'data aquisicao', 'data entrada'):
            mapa[col] = 'data_inicio'
    df_novo = df.rename(columns=mapa)
    faltando = [c for c in ('ticker', 'quantidade', 'data_inicio') if c not in df_novo.columns]
    if faltando:
        raise ValueError(f"Colunas obrigatórias não encontradas na aba Resumo: {faltando}.")
    if 'preco_medio' not in df_novo.columns and 'valor_investido' not in df_novo.columns:
        raise ValueError("A aba Resumo precisa ter 'preco_medio' OU 'valor_investido' preenchido.")
    for opcional in ('preco_medio', 'valor_investido'):
        if opcional not in df_novo.columns:
            df_novo[opcional] = None
    return df_novo[['ticker', 'quantidade', 'preco_medio', 'valor_investido', 'data_inicio']]


def _normalizar_aba_operacoes(df):
    """Colunas: ticker, tipo (compra/venda), quantidade, preco, data."""
    mapa = {}
    for col in df.columns:
        c = str(col).strip().lower()
        c = (c.replace('í', 'i').replace('é', 'e').replace('ó', 'o')
               .replace('ã', 'a').replace('ç', 'c').replace('_', ' '))
        if c in ('ticker', 'ativo', 'codigo', 'papel'):
            mapa[col] = 'ticker'
        elif c in ('tipo', 'operacao', 'movimento', 'tipo de operacao'):
            mapa[col] = 'tipo'
        elif c in ('quantidade', 'qtd', 'qtde', 'quantity'):
            mapa[col] = 'quantidade'
        elif c in ('preco', 'preco unitario', 'valor', 'valor unitario'):
            mapa[col] = 'preco'
        elif c in ('data', 'data operacao', 'data da operacao'):
            mapa[col] = 'data'
    df_novo = df.rename(columns=mapa)
    faltando = [c for c in ('ticker', 'tipo', 'quantidade', 'preco', 'data') if c not in df_novo.columns]
    if faltando:
        raise ValueError(f"Colunas obrigatórias não encontradas na aba Operações: {faltando}.")
    return df_novo[['ticker', 'tipo', 'quantidade', 'preco', 'data']]


def _construir_operacao_da_posicao_inicial(row):
    """Converte uma linha da aba Resumo (posição inicial/ponto de partida)
    numa operação de compra sintética, que entra como a PRIMEIRA operação
    do ativo, antes de qualquer movimentação da aba Operações."""
    quantidade = float(row['quantidade'])
    preco_medio = row.get('preco_medio')
    if preco_medio is None or (isinstance(preco_medio, float) and pd.isna(preco_medio)):
        valor_investido = row.get('valor_investido')
        if valor_investido is None or (isinstance(valor_investido, float) and pd.isna(valor_investido)):
            raise ValueError("informe preco_medio ou valor_investido na aba Resumo")
        preco_medio = float(valor_investido) / quantidade
    return {'data': pd.Timestamp(row['data_inicio']), 'tipo': 'compra',
            'quantidade': quantidade, 'preco': float(preco_medio)}


def processar_carteira_combinada(df_resumo, df_operacoes):
    """Uma linha final por ativo, combinando a posição inicial (aba Resumo,
    quando existir) com as movimentações (aba Operações, quando existirem).
    O Resumo funciona como ponto de partida; cada operação lançada depois
    afeta a quantidade, o preço médio e o ganho — igual um extrato real."""
    tickers_resumo = {}
    if df_resumo is not None:
        for _, row in df_resumo.iterrows():
            ticker, foi_corrigido = _normalizar_ticker(row['ticker'])
            if foi_corrigido:
                print(f"  📌 [Resumo] Ticker '{str(row['ticker']).strip()}' interpretado como '{ticker}'.")
            tickers_resumo[ticker] = row

    tickers_operacoes = {}
    if df_operacoes is not None and not df_operacoes.empty:
        tickers_normalizados = df_operacoes['ticker'].apply(lambda t: _normalizar_ticker(t)[0])
        for ticker, grupo in df_operacoes.groupby(tickers_normalizados):
            tickers_operacoes[ticker] = grupo

    todos_tickers = sorted(set(tickers_resumo) | set(tickers_operacoes))
    linhas = []

    for ticker in todos_tickers:
        moeda = 'R$' if ticker.endswith('.SA') else 'US$'
        operacoes = []
        problemas = []

        if ticker in tickers_resumo:
            try:
                operacoes.append(_construir_operacao_da_posicao_inicial(tickers_resumo[ticker]))
            except (ValueError, TypeError) as e:
                problemas.append(f"posição inicial (aba Resumo) inválida — {e}")

        n_operacoes_lancadas = 0
        if ticker in tickers_operacoes:
            for _, row in tickers_operacoes[ticker].iterrows():
                tipo_bruto = str(row['tipo']).strip().lower()
                if tipo_bruto in ('compra', 'buy', 'c'):
                    tipo = 'compra'
                elif tipo_bruto in ('venda', 'sell', 'v'):
                    tipo = 'venda'
                else:
                    problemas.append(f"tipo inválido {row['tipo']!r} numa operação (use compra/venda)")
                    continue
                try:
                    data_op = pd.Timestamp(row['data'])
                    quantidade_op = float(row['quantidade'])
                    preco_op = float(row['preco'])
                except (ValueError, TypeError):
                    problemas.append(f"operação com data/quantidade/preço ilegível (data={row['data']!r})")
                    continue
                operacoes.append({'data': data_op, 'tipo': tipo, 'quantidade': quantidade_op, 'preco': preco_op})
                n_operacoes_lancadas += 1

        if problemas:
            for p in problemas:
                print(f"  ⚠️ {ticker}: {p}")

        if not operacoes:
            linhas.append({'Ticker': ticker, 'Situação': 'erro',
                            'Ganho total': '⚠️ nenhuma posição/operação válida (ver avisos acima)'})
            continue

        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        preco_atual = tec['preco_atual'] if tec is not None else None
        divs = _dividendos_brutos_cache(ticker)

        try:
            r = avaliar_operacoes(operacoes, preco_atual=preco_atual, dividendos=divs)
        except ValueError as e:
            linhas.append({'Ticker': ticker, 'Situação': 'erro', 'Ganho total': f'⚠️ {e}'})
            continue

        situacao = 'Aberta' if r['quantidade_aberta'] > 0 else 'Encerrada'
        aviso = ' ⚠️' if problemas else ''
        origem = []
        if ticker in tickers_resumo:
            origem.append('posição inicial')
        if n_operacoes_lancadas:
            origem.append(f"{n_operacoes_lancadas} operação(ões)")
        ganho_nao_real = r['ganho_nao_realizado']

        linhas.append({
            'Ticker': ticker, 'Situação': situacao + aviso, 'Origem': ' + '.join(origem),
            'Quantidade aberta': f"{r['quantidade_aberta']:.0f}",
            'Preço médio atual': f"{moeda} {r['preco_medio_aberto']:.2f}" if r['preco_medio_aberto'] else 'N/A',
            'Ganho realizado': f"{moeda} {r['ganho_realizado']:+.2f}",
            'Ganho não realizado': f"{moeda} {ganho_nao_real:+.2f}" if ganho_nao_real is not None else 'N/A',
            'Renda recebida': f"{moeda} {r['renda_recebida']:.2f}",
            'Ganho total': f"{moeda} {r['ganho_total']:+.2f}",
            '_moeda': moeda, '_ganho_total_num': r['ganho_total'],
            '_ganho_realizado_num': r['ganho_realizado'],
        })

    return pd.DataFrame(linhas)


def _achar_aba(abas_dict, nomes_possiveis):
    """Procura uma aba pelo nome, tolerando maiúsculas/minúsculas e acento."""
    def normalizar_nome(n):
        n = n.strip().lower()
        return n.replace('í', 'i').replace('é', 'e').replace('ç', 'c').replace('õ', 'o').replace('ã', 'a')
    alvo = {normalizar_nome(n) for n in nomes_possiveis}
    for nome_real, df in abas_dict.items():
        if normalizar_nome(nome_real) in alvo:
            return df
    return None


# --- Painel: importar carteira (abas Resumo + Operações) e calcular ---
origem_planilha = widgets.Dropdown(
    options=[('Upload de arquivo (Excel/CSV)', 'upload'), ('Google Sheets (link)', 'sheets')],
    value='upload', description='Fonte:', style={'description_width': 'initial'},
)
link_sheets = widgets.Text(
    value='', description='Link da planilha:',
    placeholder='https://docs.google.com/spreadsheets/d/...',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
botao_importar = widgets.Button(description="📂 Importar e calcular", button_style='success', icon='upload')
saida_carteira_planilha = widgets.Output()

def _ler_todas_as_abas(origem):
    if origem == 'upload':
        from google.colab import files
        print("Selecione o arquivo (Excel .xlsx ou CSV) na janela que vai abrir...")
        enviados = files.upload()
        if not enviados:
            return None
        nome_arquivo = list(enviados.keys())[0]
        if nome_arquivo.lower().endswith('.csv'):
            return {'Operações': pd.read_csv(nome_arquivo)}  # CSV = 1 tabela só, tratada como Operações
        return pd.read_excel(nome_arquivo, sheet_name=None)
    else:
        if not link_sheets.value.strip():
            print("⚠️ Cole o link da planilha do Google Sheets no campo acima.")
            return None
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        planilha = gc.open_by_url(link_sheets.value.strip())
        return {aba.title: pd.DataFrame(aba.get_all_records()) for aba in planilha.worksheets()}


def _importar_carteira_click(b):
    with saida_carteira_planilha:
        clear_output()
        try:
            abas = _ler_todas_as_abas(origem_planilha.value)
            if abas is None:
                return
            if not abas:
                print("⚠️ Nenhuma aba encontrada na planilha.")
                return

            print(f"Abas encontradas: {', '.join(abas.keys())}\n")

            df_resumo_bruto = _achar_aba(abas, ['Resumo', 'Resumo da carteira', 'Posicao', 'Posição'])
            df_operacoes_bruto = _achar_aba(abas, ['Operações', 'Operacoes', 'Movimentações', 'Movimentacoes'])

            df_resumo = _normalizar_aba_resumo(df_resumo_bruto) if df_resumo_bruto is not None else None
            df_operacoes = _normalizar_aba_operacoes(df_operacoes_bruto) if df_operacoes_bruto is not None else None

            if df_resumo is None and df_operacoes is None:
                print("⚠️ Não encontrei nenhuma aba chamada 'Resumo' ou 'Operações' (nem variações). "
                      "Renomeie as abas do arquivo e tente de novo.")
                return

            print(f"Aba Resumo: {'não encontrada' if df_resumo is None else f'{len(df_resumo)} ativo(s)'}")
            print(f"Aba Operações: {'não encontrada' if df_operacoes is None else f'{len(df_operacoes)} operação(ões)'}\n")

            resultado = processar_carteira_combinada(df_resumo, df_operacoes)
            colunas_exibir = [c for c in resultado.columns if not c.startswith('_')]

            situacao = resultado['Situação'].astype(str) if 'Situação' in resultado.columns else pd.Series(dtype=str)
            abertas = resultado[situacao.str.startswith('Aberta')]
            encerradas = resultado[situacao.str.startswith('Encerrada')]
            com_erro = resultado[situacao == 'erro']

            print(f"\n📌 CARTEIRA ATUAL — {len(abertas)} posição(ões) em aberto "
                  f"(inclui ativos novos, exclui os totalmente vendidos):")
            if not abertas.empty:
                display(abertas[colunas_exibir])
            else:
                print("   (nenhuma posição em aberto)")

            if not encerradas.empty:
                print(f"\n📁 POSIÇÕES ENCERRADAS NO PERÍODO — {len(encerradas)} ativo(s) totalmente "
                      f"vendido(s) (fora da carteira atual, histórico do ganho já realizado):")
                display(encerradas[colunas_exibir])

            if not com_erro.empty:
                print(f"\n⚠️ {len(com_erro)} ativo(s) com erro — ver avisos acima e a tabela abaixo:")
                display(com_erro[colunas_exibir])

            if '_ganho_total_num' in resultado.columns:
                print("\n📊 TOTAIS POR MOEDA:")
                for rotulo, subset in [
                    ("Carteira atual — ganho total (inclui realizado de vendas parciais + não realizado + renda)", abertas),
                    ("Posições encerradas — ganho total (100% realizado)", encerradas),
                ]:
                    df_valido = subset[subset['_ganho_total_num'].notna()]
                    if df_valido.empty:
                        continue
                    for moeda, grupo in df_valido.groupby('_moeda'):
                        total_ganho = grupo['_ganho_total_num'].sum()
                        print(f"   {rotulo}\n      {moeda}: {moeda} {total_ganho:+,.2f}")

                if '_ganho_realizado_num' in resultado.columns:
                    df_valido_real = resultado[resultado['_ganho_realizado_num'].notna()]
                    if not df_valido_real.empty:
                        print("\n💰 GANHO JÁ REALIZADO NO TOTAL (soma vendas parciais + posições encerradas, "
                              "por moeda):")
                        for moeda, grupo in df_valido_real.groupby('_moeda'):
                            total_realizado = grupo['_ganho_realizado_num'].sum()
                            print(f"   {moeda}: {moeda} {total_realizado:+,.2f}")

                print("\n📌 Os totais são calculados separadamente por moeda (R$ e US$), sem conversão")
                print("   cambial entre elas.")
        except ImportError:
            print("⚠️ Este recurso depende de bibliotecas específicas do Google Colab "
                  "(google.colab, gspread) — só funciona quando executado no Colab, não localmente.")
        except Exception as e:
            print(f"⚠️ Erro ao processar a planilha: {e}")

botao_importar.on_click(_importar_carteira_click)
display(widgets.HTML("<h3>9.3 Ganho consolidado da carteira (via planilha)</h3>"))
display(widgets.HTML(
    "<p>Importa a carteira de um arquivo Excel/CSV ou Google Sheets, combinando duas abas:</p>"
    "<ul>"
    "<li><b>Resumo</b> — a posição inicial de cada ativo (ponto de partida): "
    "<code>ticker</code>, <code>quantidade</code>, <code>preco_medio</code> (ou "
    "<code>valor_investido</code>), <code>data_inicio</code>.</li>"
    "<li><b>Operações</b> — as movimentações feitas a partir dali: <code>ticker</code>, "
    "<code>tipo</code> (compra/venda), <code>quantidade</code>, <code>preco</code>, <code>data</code>.</li>"
    "</ul>"
    "<p>Um ativo pode ter só Resumo, só Operações, ou as duas — as operações são sempre "
    "aplicadas EM CIMA da posição inicial, ajustando quantidade, preço médio e ganho, "
    "igual um extrato.</p>"
))
display(widgets.VBox([origem_planilha, link_sheets, botao_importar, saida_carteira_planilha]))


## 📐 10. Backtesting do sinal de timing (técnico)

Reavalia retroativamente o componente **técnico** do motor de pontuação (tendência, RSI, distância do período) sobre o histórico de cada ativo da carteira, comparando o sinal que teria sido dado no passado com o retorno realmente observado depois — F1-score (macro, 3 classes: compra/manter/venda).

Todos os parâmetros (horizonte de avaliação, limiar de retorno, e os limiares de pontuação que definem compra/venda em dois cenários) são **selecionáveis nos controles abaixo**, sem precisar editar código — os valores padrão refletem os parâmetros definidos na Seção 3.1 do Relatório do Projeto.

**Limitação conhecida**: nem o Yahoo Finance nem o Fundamentus (Seção 5) fornecem uma série histórica gratuita de fundamentos (P/VP, vacância, dividend yield de datas passadas) — os dois só dão o valor de **hoje**. Por isso este backtest avalia só o componente técnico, não o motor completo usado no dia a dia da ferramenta. É uma validação parcial, não do sistema inteiro. Para compensar isso, a Seção mostra, ao final, o retrato fundamentalista **atual** de cada FII (via Fundamentus) como contexto complementar — não como parte do cálculo retroativo — e uma interpretação em linguagem simples do resultado técnico.

In [ ]:
#@title 📐 10. Backtesting do sinal de timing (técnico)

def _indicadores_de_serie(fechamento_bruto):
    """Mesma lógica de cálculo de indicadores técnicos da Seção 4, mas
    recebendo a série de preços diretamente (em vez de baixar por ticker) —
    permite aplicar sobre um RECORTE truncado do histórico, "congelando"
    uma data de corte no passado, sem espiar dado futuro (look-ahead bias)."""
    fechamento = fechamento_bruto.dropna()
    if len(fechamento) < 30:
        return None
    fechamento, n_outliers, n_truncados = limpar_outliers_precos(fechamento)
    if len(fechamento) < 30:
        return None
    preco_atual = float(fechamento.iloc[-1])
    sma200 = fechamento.rolling(200).mean().iloc[-1] if len(fechamento) >= 200 else np.nan
    rsi14 = calcular_rsi(fechamento, 14).iloc[-1]
    maxima_periodo = fechamento.quantile(0.95)
    minima_periodo = fechamento.quantile(0.05)
    dist_maxima = (preco_atual - maxima_periodo) / maxima_periodo * 100
    dist_minima = (preco_atual - minima_periodo) / minima_periodo * 100
    return {
        'preco_atual': preco_atual,
        'sma200': float(sma200) if not pd.isna(sma200) else None,
        'rsi14': float(rsi14) if not pd.isna(rsi14) else None,
        'dist_maxima_pct': float(dist_maxima),
        'dist_minima_pct': float(dist_minima),
    }


def _pontuar_tecnico_backtest(tec):
    """Componente técnico do motor de pontuação, isolado do fundamentalista —
    é o único componente reconstruível retroativamente, já que o Yahoo Finance
    não fornece fundamentos históricos (P/L, P/VP de datas passadas)."""
    pontos = 0
    if tec['sma200'] is not None:
        pontos += 1 if tec['preco_atual'] > tec['sma200'] else -1
    if tec['rsi14'] is not None:
        if tec['rsi14'] < 40:
            pontos += 1
        elif tec['rsi14'] > 70:
            pontos -= 1
    if tec['dist_minima_pct'] < 15:
        pontos += 1
    if tec['dist_maxima_pct'] > -5:
        pontos -= 1
    return pontos


def _classe_a_partir_de_pontos(pontos, limiar_compra=3, limiar_venda=-2):
    if pontos >= limiar_compra:
        return 'compra'
    if pontos <= limiar_venda:
        return 'venda'
    return 'manter'


def _rotular_retorno_futuro(fechamento, data_corte, horizonte_dias, limiar):
    pos_corte = fechamento.index.searchsorted(data_corte)
    if pos_corte >= len(fechamento):
        return None
    preco_corte = float(fechamento.iloc[pos_corte])
    data_alvo = data_corte + pd.Timedelta(days=horizonte_dias)
    pos_futuro = fechamento.index.searchsorted(data_alvo)
    if pos_futuro >= len(fechamento):
        return None
    preco_futuro = float(fechamento.iloc[pos_futuro])
    retorno = (preco_futuro - preco_corte) / preco_corte
    if retorno > limiar:
        return 'compra'
    if retorno < -limiar:
        return 'venda'
    return 'manter'


def _yield_ultimos_12m_backtest(dividendos_ate_data, preco):
    """Yield 'trailing' (12m até a data de corte) — usado como previsão
    'ingênua' (persistência) do yield que será pago nos 12 meses seguintes."""
    if dividendos_ate_data is None or dividendos_ate_data.empty or not preco:
        return None
    hoje = dividendos_ate_data.index.max()
    ult_12m = dividendos_ate_data[dividendos_ate_data.index > hoje - pd.Timedelta(days=365)]
    if ult_12m.empty:
        return None
    return float(ult_12m.sum()) / preco


def _baixar_fechamento_cache(ticker, periodo='5y'):
    """Busca o histórico de preço de um ticker com cache (mesmo padrão já
    usado pra dividendos nesta seção) — evita rebuscar o mesmo ticker
    quando o backtest é rodado várias vezes com horizontes diferentes
    (sensibilidade ao horizonte, mais abaixo nesta seção)."""
    chave = f"precos_backtest:{ticker}:{periodo}"
    fechamento = _cache_get(chave)
    if fechamento is not None:
        return fechamento
    hist = yf.download(ticker, period=periodo, progress=False)
    if hist.empty:
        fechamento = pd.Series(dtype=float)
    else:
        fechamento = hist['Close']
        if isinstance(fechamento, pd.DataFrame):
            fechamento = fechamento.squeeze(axis=1)
        fechamento = fechamento.dropna()
    _cache_set(chave, fechamento)
    return fechamento


def executar_backtest_ticker(ticker, periodo='5y', passo_dias=30, horizonte_dias=90, limiar_retorno=0.05,
                              limiar_compra=3, limiar_venda=-2, dividendos=None, fechamento_completo=None):
    """Roda o backtest retroativo de um ticker: percorre datas de corte a
    cada `passo_dias`, calcula o sinal técnico só com dado disponível até
    ali, e compara com o retorno realmente observado `horizonte_dias`
    depois. Parâmetros padrão: horizonte de 90 dias e limiar de ±5%,
    conforme definido na Seção 3.1 (Objetivo SMART) do Relatório do Projeto.
    Também registra, em cada ponto de corte, o yield trailing (12m até ali)
    como previsão 'ingênua' e o yield efetivo pago nos 12 meses seguintes —
    usados depois pra calcular MAE/R² da previsão de renda.

    `fechamento_completo` pode ser passado pronto (já buscado antes) pra
    evitar rebuscar o mesmo ticker ao testar múltiplos horizontes."""
    if fechamento_completo is None:
        fechamento_completo = _baixar_fechamento_cache(ticker, periodo)
    if fechamento_completo.empty:
        return []

    if dividendos is None:
        dividendos = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
    elif dividendos.index.tz is not None:
        dividendos = dividendos.copy()
        dividendos.index = dividendos.index.tz_localize(None)

    registros = []
    for idx in range(200, len(fechamento_completo) - 1, passo_dias):
        data_corte = fechamento_completo.index[idx]
        fechamento_ate_corte = fechamento_completo.iloc[:idx + 1]
        tec = _indicadores_de_serie(fechamento_ate_corte)
        if tec is None:
            continue
        classe_real = _rotular_retorno_futuro(fechamento_completo, data_corte, horizonte_dias, limiar_retorno)
        if classe_real is None:
            continue
        pontos_tecnicos = _pontuar_tecnico_backtest(tec)
        classe_prevista = _classe_a_partir_de_pontos(pontos_tecnicos, limiar_compra, limiar_venda)

        dividendos_ate_corte = dividendos[dividendos.index <= data_corte]
        dy_trailing = _yield_ultimos_12m_backtest(dividendos_ate_corte, tec['preco_atual'])
        data_futuro_12m = data_corte + pd.Timedelta(days=365)
        dividendos_futuros = dividendos[(dividendos.index > data_corte) & (dividendos.index <= data_futuro_12m)]
        dy_futuro = float(dividendos_futuros.sum()) / tec['preco_atual'] if tec['preco_atual'] else None

        registros.append({'ticker': ticker, 'data_corte': data_corte, 'pontos_tecnicos': pontos_tecnicos,
                           'classe_prevista': classe_prevista, 'classe_real': classe_real,
                           'dy_trailing_previsto': dy_trailing, 'dy_futuro_realizado': dy_futuro})
    return registros


def reclassificar(registros, limiar_compra=3, limiar_venda=-2):
    """Reclassifica um conjunto de registros já calculados, com limiares
    diferentes, sem precisar baixar dado nem recalcular indicadores de novo
    — só reaplica a régua de classificação sobre a pontuação já guardada."""
    novos = []
    for r in registros:
        nova_classe = _classe_a_partir_de_pontos(r['pontos_tecnicos'], limiar_compra, limiar_venda)
        novos.append({**r, 'classe_prevista': nova_classe})
    return novos


def calcular_f1_macro(registros):
    classes = ('compra', 'manter', 'venda')
    por_classe = {}
    for c in classes:
        tp = sum(1 for r in registros if r['classe_prevista'] == c and r['classe_real'] == c)
        fp = sum(1 for r in registros if r['classe_prevista'] == c and r['classe_real'] != c)
        fn = sum(1 for r in registros if r['classe_prevista'] != c and r['classe_real'] == c)
        precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0.0
        por_classe[c] = {'precisao': precisao, 'recall': recall, 'f1': f1, 'n_real': tp + fn}
    f1_macro = sum(m['f1'] for m in por_classe.values()) / len(classes) if registros else 0.0
    return f1_macro, por_classe


def calcular_mae_r2_renda(registros):
    """MAE e R² comparando o yield trailing (previsão 'ingênua' de
    persistência) contra o yield efetivo realmente pago nos 12 meses
    seguintes à data de corte — mesma lógica de backtesting.py (pacote)."""
    pares = [(r['dy_trailing_previsto'], r['dy_futuro_realizado']) for r in registros
             if r.get('dy_trailing_previsto') is not None and r.get('dy_futuro_realizado') is not None]
    if not pares:
        return {'mae': None, 'r2': None, 'n': 0}
    previstos = np.array([p[0] for p in pares])
    reais = np.array([p[1] for p in pares])
    mae = float(np.mean(np.abs(previstos - reais)))
    media_real = reais.mean()
    ss_tot = float(np.sum((reais - media_real) ** 2))
    ss_res = float(np.sum((reais - previstos) ** 2))
    r2 = (1 - ss_res / ss_tot) if ss_tot > 0 else None
    return {'mae': mae, 'r2': r2, 'n': len(pares)}


# --- Painel: parâmetros selecionáveis + rodar o backtest ---
horizonte_backtest = widgets.IntSlider(value=90, min=30, max=180, step=15,
    description='Horizonte (dias):', style={'description_width': 'initial'})
limiar_retorno_backtest = widgets.FloatSlider(value=0.05, min=0.02, max=0.15, step=0.01,
    description='Limiar de retorno (±):', readout_format='.0%', style={'description_width': 'initial'})

limiar_compra_a = widgets.IntSlider(value=3, min=-3, max=3, step=1,
    description='Cenário A — limiar de compra:', style={'description_width': 'initial'})
limiar_venda_a = widgets.IntSlider(value=-2, min=-3, max=3, step=1,
    description='Cenário A — limiar de venda:', style={'description_width': 'initial'})
limiar_compra_b = widgets.IntSlider(value=2, min=-3, max=3, step=1,
    description='Cenário B — limiar de compra:', style={'description_width': 'initial'})
limiar_venda_b = widgets.IntSlider(value=-2, min=-3, max=3, step=1,
    description='Cenário B — limiar de venda:', style={'description_width': 'initial'})

botao_backtest = widgets.Button(description="📐 Rodar backtesting", button_style='warning', icon='line-chart')
saida_backtest = widgets.Output()

def _imprimir_resultado_cenario(nome_cenario, registros):
    f1_macro, por_classe = calcular_f1_macro(registros)
    print(f"\n📊 {nome_cenario}: F1-score macro = {f1_macro:.3f}")
    for c, m in por_classe.items():
        print(f"   {c}: precisão={m['precisao']:.2f}  recall={m['recall']:.2f}  f1={m['f1']:.2f}  (n={m['n_real']})")
    return f1_macro



def _interpretar_em_linguagem_simples(registros, f1_macro, por_classe):
    """Traduz o resultado técnico do backtest para linguagem simples, sem
    jargão estatístico — pensado para o perfil de conhecimento intermediário
    (Persona 2, Seção 2.2 do Relatório), que pode não ter familiaridade com
    F1-score/precisão/recall."""
    from collections import Counter
    n_total = len(registros)
    if n_total == 0:
        return
    contagem_real = Counter(r['classe_real'] for r in registros)
    classe_maj, n_maj = contagem_real.most_common(1)[0]
    p_maj = n_maj / n_total
    r_maj = 1.0
    f1_maj_classe = 2 * p_maj * r_maj / (p_maj + r_maj) if (p_maj + r_maj) > 0 else 0
    f1_majoritaria = f1_maj_classe / 3

    recall_compra = por_classe.get('compra', {}).get('recall', 0)
    recall_venda = por_classe.get('venda', {}).get('recall', 0)

    print("\n" + "=" * 66)
    print("💬 O QUE ISSO SIGNIFICA, EM PALAVRAS SIMPLES:")
    print("=" * 66)
    print(f"\nDas vezes em que REALMENTE teria valido a pena COMPRAR (olhando pra trás,")
    print(f"sabendo o que aconteceu depois), o sinal técnico identificou corretamente")
    print(f"apenas {recall_compra*100:.0f}% delas.")
    print(f"\nDas vezes em que REALMENTE teria valido a pena VENDER, identificou")
    print(f"corretamente apenas {recall_venda*100:.0f}% delas.")
    print(f"\nComparando com um critério \"preguiçoso\" que nunca sugere nada (sempre")
    print(f"\"manter\"), o sinal técnico pontuou {f1_macro:.2f} contra {f1_majoritaria:.2f}")
    print(f"do critério preguiçoso — diferença de {(f1_macro - f1_majoritaria):+.2f}.")
    if abs(f1_macro - f1_majoritaria) < 0.05:
        print(f"\n👉 Isso indica que, sozinho, o sinal técnico teve desempenho muito parecido")
        print(f"   com simplesmente não fazer nada. Não deve ser usado como única base pra")
        print(f"   decidir comprar ou vender — funciona melhor como mais um dado a considerar,")
        print(f"   junto com os fundamentos (Seção 5) e o contexto do ativo.")
    elif f1_macro > f1_majoritaria:
        print(f"\n👉 O sinal técnico teve desempenho melhor que simplesmente não fazer nada,")
        print(f"   mas ainda está longe de ser perfeito — use como um dado a mais na sua")
        print(f"   análise, não como resposta pronta.")
    else:
        print(f"\n👉 Neste teste, o sinal técnico teve desempenho PIOR que simplesmente não")
        print(f"   fazer nada — um sinal de que, pelo menos neste histórico e com estes")
        print(f"   parâmetros, ele não ajudou a prever o que aconteceria depois.")


def _exibir_contexto_fundamentus_atual(tickers):
    """Mostra o retrato ATUAL (não histórico) dos fundamentos de FIIs via
    Fundamentus, como contexto complementar ao backtest técnico acima — não
    faz parte do cálculo retroativo, porque o Fundamentus só fornece o valor
    de HOJE desses indicadores, não uma série histórica gratuita (ver nota
    desta seção)."""
    linhas_fii = []
    for ticker in tickers:
        f = buscar_fundamentos_fii_fundamentus(ticker)
        if f is not None:
            linhas_fii.append((ticker, f))

    if not linhas_fii:
        return

    print("\n" + "=" * 66)
    print("🏢 CONTEXTO FUNDAMENTALISTA ATUAL DOS FIIs (via Fundamentus, hoje)")
    print("=" * 66)
    print("Não faz parte do backtest acima (que é só técnico e retroativo) — é o")
    print("retrato mais recente disponível, para complementar a leitura histórica")
    print("com a situação atual de cada fundo.\n")
    for ticker, f in linhas_fii:
        pvp = f.get('pvp')
        dy = f.get('dividend_yield_fundamentus')
        vac = f.get('vacancia_media')
        segm = f.get('segmento') or 'N/D'
        pvp_txt = f"P/VP={pvp:.2f}" if pvp is not None else "P/VP=N/D"
        dy_txt = f"Dividend Yield={dy*100:.1f}%" if dy is not None else "Dividend Yield=N/D"
        # FII de papel (sem imóveis físicos) não tem vacância no sentido usual.
        # Três sinais combinados (nenhum sozinho é sempre confiável — o
        # RECR11.SA já voltou a mostrar qtd_imoveis preenchido incorretamente
        # depois da primeira correção, Seção 4.3 do Relatório): segmento
        # classificado como papel/recebíveis/títulos, qtd_imoveis ausente/
        # zero, ou vacância extrema (100% exato — implausível num FII de
        # tijolo real).
        segmento_lower = segm.lower()
        segmento_indica_papel = any(
            p in segmento_lower for p in ('papel', 'recebív', 'recebiv', 'título', 'titulo', 'renda fixa'))
        qtd_imoveis_indica_papel = f.get('qtd_imoveis') in (None, 0)
        vacancia_extrema_suspeita = vac is not None and vac >= 0.999
        eh_fii_de_papel = qtd_imoveis_indica_papel or segmento_indica_papel or vacancia_extrema_suspeita
        vac_txt = "Vacância=N/A (FII de papel)" if eh_fii_de_papel else (
            f"Vacância={vac*100:.1f}%" if vac is not None else "Vacância=N/D")
        print(f"  {ticker} ({segm}): {pvp_txt} | {dy_txt} | {vac_txt}")


def _rodar_backtest_click(b):
    with saida_backtest:
        clear_output()
        # Mapa ticker -> tipo de ativo, usado na quebra de F1 por categoria mais abaixo.
        tipo_por_ticker = {}
        for t in parse_tickers(caixa_acoes.value):
            tipo_por_ticker[t] = 'Ação B3'
        for t in parse_tickers(caixa_fiis.value):
            tipo_por_ticker[t] = 'FII'
        for t in parse_tickers(caixa_acoes_us.value):
            tipo_por_ticker[t] = 'Ação EUA'
        for t in parse_tickers(caixa_etf_br.value):
            tipo_por_ticker[t] = 'ETF B3'
        for t in parse_tickers(caixa_etf_us.value):
            tipo_por_ticker[t] = 'ETF EUA'
        todos_tickers = list(tipo_por_ticker.keys())
        if not todos_tickers:
            print("⚠️ Nenhum ticker configurado na Seção 2.")
            return

        horizonte = horizonte_backtest.value
        limiar_ret = limiar_retorno_backtest.value
        lc_a, lv_a = limiar_compra_a.value, limiar_venda_a.value
        lc_b, lv_b = limiar_compra_b.value, limiar_venda_b.value

        print(f"Rodando backtest (horizonte {horizonte} dias, limiar de retorno ±{limiar_ret*100:.0f}%) "
              f"para {len(todos_tickers)} ativo(s)... isso pode demorar um pouco.\n")
        todos_registros = []
        for ticker in todos_tickers:
            # Preço e dividendos com cache (mesmo padrão já usado no resto do notebook) —
            # o cache de preço evita rebuscar o mesmo ticker na sensibilidade ao horizonte,
            # mais abaixo, que roda o backtest de novo com horizontes diferentes.
            fechamento_ticker = _baixar_fechamento_cache(ticker)
            chave_cache_div = f"dividendos:{ticker}"
            divs_ticker = _cache_get(chave_cache_div)
            if divs_ticker is None:
                try:
                    divs_ticker = yf.Ticker(ticker).dividends
                except Exception:
                    divs_ticker = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
                _cache_set(chave_cache_div, divs_ticker)

            # Roda uma vez com os limiares do Cenário A — os pontos_tecnicos ficam guardados
            # no registro, permitindo reclassificar pro Cenário B (e pro sweep, mais abaixo)
            # sem baixar o dado de novo.
            registros = executar_backtest_ticker(ticker, horizonte_dias=horizonte, limiar_retorno=limiar_ret,
                                                  limiar_compra=lc_a, limiar_venda=lv_a, dividendos=divs_ticker,
                                                  fechamento_completo=fechamento_ticker)
            for r in registros:
                r['tipo_ativo'] = tipo_por_ticker[ticker]
            print(f"  {ticker}: {len(registros)} ponto(s) de corte avaliado(s)")
            todos_registros.extend(registros)

        if not todos_registros:
            print("\n⚠️ Não foi possível gerar nenhum ponto de backtest (histórico curto demais pra algum ativo).")
            return

        f1_a, por_classe_a = calcular_f1_macro(todos_registros)
        _imprimir_resultado_cenario(f"Cenário A (compra ≥ {lc_a}, venda ≤ {lv_a})", todos_registros)

        registros_b = reclassificar(todos_registros, limiar_compra=lc_b, limiar_venda=lv_b)
        f1_b, por_classe_b = calcular_f1_macro(registros_b)
        _imprimir_resultado_cenario(f"Cenário B (compra ≥ {lc_b}, venda ≤ {lv_b})", registros_b)

        print(f"\n🔍 Comparação: F1 macro Cenário A = {f1_a:.3f}  |  F1 macro Cenário B = {f1_b:.3f}")
        print(f"   Diferença (B − A): {f1_b - f1_a:+.3f}")

        # Interpretação em linguagem simples do cenário A (o padrão/principal)
        _interpretar_em_linguagem_simples(todos_registros, f1_a, por_classe_a)

        # --- Previsão de renda: MAE/R² (persistência do yield trailing vs. realizado) ---
        resultado_renda = calcular_mae_r2_renda(todos_registros)
        print("\n" + "=" * 66)
        print("💰 PREVISÃO DE RENDA (yield trailing como previsão 'ingênua')")
        print("=" * 66)
        if resultado_renda['n'] == 0:
            print("Não há pares suficientes (yield trailing + yield futuro realizado) pra")
            print("calcular MAE/R² — comum se os ativos da carteira pagarem pouco ou")
            print("irregularmente dividendo/rendimento no histórico disponível.")
        else:
            mae_txt = f"{resultado_renda['mae']*100:.2f} pontos percentuais" if resultado_renda['mae'] is not None else "N/D"
            r2_txt = f"{resultado_renda['r2']:.3f}" if resultado_renda['r2'] is not None else "N/D (variância real zero)"
            print(f"N = {resultado_renda['n']} pontos de corte com par válido (yield trailing, yield futuro)")
            print(f"MAE (erro médio absoluto): {mae_txt}")
            print(f"R² (quanto da variação do yield futuro a previsão ingênua explica): {r2_txt}")
            print(f"\n💬 Em palavras simples: usar o yield dos últimos 12 meses como \"aposta\" pro")
            print(f"   yield dos PRÓXIMOS 12 meses erra, em média, {mae_txt} — e um R² baixo (perto")
            print(f"   de 0 ou negativo) indica que essa aposta simples não captura bem as mudanças")
            print(f"   reais de renda; um R² mais alto (perto de 1) indica que o yield passado é um")
            print(f"   bom indício do yield futuro, para os ativos e período testados.")

        # --- Quebra do F1-score por tipo de ativo ---
        # O F1 macro geral é uma média de ativos bem diferentes entre si — pode esconder
        # o sinal funcionando bem numa categoria e mal noutra.
        print("\n" + "=" * 66)
        print("📂 F1-SCORE POR TIPO DE ATIVO (Cenário A)")
        print("=" * 66)
        for tipo in sorted(set(tipo_por_ticker.values())):
            registros_tipo = [r for r in todos_registros if r['tipo_ativo'] == tipo]
            if not registros_tipo:
                continue
            f1_tipo, _ = calcular_f1_macro(registros_tipo)
            print(f"  {tipo}: F1 macro = {f1_tipo:.3f}  (n={len(registros_tipo)})")

        # --- Sweep sistemático de limiares ---
        # Em vez de só os 2 cenários escolhidos à mão acima, testa uma grade de combinações
        # de limiar de compra/venda sobre os MESMOS registros já calculados (reclassificar()
        # não rebusca dado, só reaplica a régua de classificação) — mostra se existe uma
        # combinação melhor que a testada, ou se o teto é baixo mesmo com qualquer limiar.
        print("\n" + "=" * 66)
        print("🔬 SWEEP SISTEMÁTICO DE LIMIARES (compra: 1 a 4 · venda: -1 a -4)")
        print("=" * 66)
        resultados_sweep = []
        for lc_sweep in range(1, 5):
            for lv_sweep in range(-4, 0):
                registros_sweep = reclassificar(todos_registros, limiar_compra=lc_sweep, limiar_venda=lv_sweep)
                f1_sweep, _ = calcular_f1_macro(registros_sweep)
                resultados_sweep.append((lc_sweep, lv_sweep, f1_sweep))
        resultados_sweep.sort(key=lambda x: -x[2])
        print(f"Testadas {len(resultados_sweep)} combinações. Top 5 (compra, venda, F1 macro):")
        for lc_r, lv_r, f1_r in resultados_sweep[:5]:
            print(f"  compra≥{lc_r}, venda≤{lv_r}: F1 = {f1_r:.3f}")
        pior = resultados_sweep[-1]
        melhor = resultados_sweep[0]
        print(f"\nFaixa de F1 no sweep inteiro: {pior[2]:.3f} (pior: compra≥{pior[0]}, venda≤{pior[1]}) "
              f"a {melhor[2]:.3f} (melhor: compra≥{melhor[0]}, venda≤{melhor[1]})")
        if melhor[2] - f1_a < 0.03:
            print("👉 Nenhuma combinação testada supera de forma relevante o Cenário A original —")
            print("   o teto do sinal técnico parece ser mesmo baixo neste histórico, não uma questão")
            print("   de limiar mal escolhido.")
        else:
            print(f"👉 A melhor combinação do sweep (compra≥{melhor[0]}, venda≤{melhor[1]}) supera o Cenário A")
            print(f"   em {melhor[2] - f1_a:+.3f} — pode valer adotar esses limiares como novo padrão.")

        # --- Sensibilidade ao horizonte ---
        # Reaproveita preço e dividendo já em cache (buscados uma vez, acima) — só recalcula
        # os pontos de corte/rótulos pra cada horizonte, sem nova chamada de rede.
        print("\n" + "=" * 66)
        print("⏱️ SENSIBILIDADE AO HORIZONTE (mesmos limiares do Cenário A)")
        print("=" * 66)
        for h_teste in (30, 60, 90, 120, 180):
            registros_h = []
            for ticker in todos_tickers:
                fechamento_ticker = _baixar_fechamento_cache(ticker)
                divs_ticker = _cache_get(f"dividendos:{ticker}")
                if divs_ticker is None:
                    divs_ticker = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
                regs_h = executar_backtest_ticker(ticker, horizonte_dias=h_teste, limiar_retorno=limiar_ret,
                                                   limiar_compra=lc_a, limiar_venda=lv_a, dividendos=divs_ticker,
                                                   fechamento_completo=fechamento_ticker)
                registros_h.extend(regs_h)
            if not registros_h:
                continue
            f1_h, _ = calcular_f1_macro(registros_h)
            marcador_atual = "  ← horizonte atual" if h_teste == horizonte else ""
            print(f"  {h_teste:>3} dias: F1 macro = {f1_h:.3f}  (n={len(registros_h)}){marcador_atual}")

        # Contexto fundamentalista atual (Fundamentus) — complementar, não retroativo
        _exibir_contexto_fundamentus_atual(todos_tickers)

        print("\n📌 Este backtest avalia SÓ o componente técnico do motor — nem o Yahoo Finance")
        print("   nem o Fundamentus fornecem fundamentos históricos gratuitos (ver nota acima) —")
        print("   é uma validação parcial, não do motor completo.")
        print("   Todos os parâmetros acima são ajustáveis nos controles desta seção, sem editar código.")

botao_backtest.on_click(_rodar_backtest_click)
display(widgets.HTML("<h3>10. Backtesting do sinal de timing</h3>"))
display(widgets.HTML("<p>Reavalia retroativamente o componente técnico do motor de pontuação sobre "
                      "o histórico de cada ativo da carteira (Seção 2), comparando o sinal que teria "
                      "sido dado no passado com o retorno realmente observado depois. Ajuste os "
                      "parâmetros abaixo e compare dois cenários de limiar lado a lado.</p>"))
display(widgets.VBox([
    horizonte_backtest, limiar_retorno_backtest,
    widgets.HTML("<b>Cenário A</b>"), limiar_compra_a, limiar_venda_a,
    widgets.HTML("<b>Cenário B</b>"), limiar_compra_b, limiar_venda_b,
    botao_backtest, saida_backtest,
]))


## 📌 Notas finais

- Os critérios usados aqui (Seção 6) são **regras de bolso genéricas** de mercado — muito usadas como ponto de
  partida, mas nenhuma delas garante retorno nem substitui análise mais profunda (relatórios setoriais, balanços
  completos, contexto macroeconômico).
- Dados fundamentalistas via Yahoo Finance para tickers da B3 às vezes vêm incompletos — quando faltar, o indicador
  correspondente simplesmente não pontua (é tratado como neutro), o que pode enviesar o resultado. Se um sinal
  parecer estranho, vale conferir os números na fonte original (RI da empresa/fundo, B3, etc.).
- Este notebook **não é uma recomendação de investimento** nem substitui um assessor/consultor licenciado (CVM).
